# YRBS LGBT Youth Mental Health Risk Stratification

Analysis of Youth Risk Behavior Surveillance System (YRBS) data studying mental health risk among sexual and gender minority (SGM) youth.

## 1. Setup and Data Loading

In [4]:
from typing import Any, List, Dict
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, recall_score, precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.validation import check_is_fitted
from sklearn.utils import check_random_state
from sklearn.linear_model import BayesianRidge
from lightgbm import LGBMClassifier
import lightgbm as lgb
import shap
from collections import defaultdict
from tqdm.auto import tqdm
import time

In [5]:

raw_data = pd.read_csv("/content/drive/MyDrive/pods_project1/new_data_project_2/XXHq.csv")
print("Shape:", raw_data.shape)
raw_data.head(2)

Shape: (20103, 117)


/tmp/ipykernel_23630/3259062932.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv("/content/drive/MyDrive/pods_project1/new_data_project_2/XXHq.csv")


,site,raceeth,q6orig,q7orig,record,orig_rec,q1,q2,q3,q4,...,q102,q103,q104,q105,q106,q107,BMIPCT,weight,stratum,psu
0,XX,NaN,505,180,1.0,NaN,3.0,1.0,1.0,NaN,...,2.0,3.0,3.0,2.0,2.0,2.0,97.08,0.86,103.0,16294.0
1,XX,5.0,N N,233,2.0,NaN,4.0,2.0,1.0,2.0,...,2.0,2.0,5.0,2.0,1.0,2.0,NaN,0.89,103.0,16294.0


In [6]:
# Variable dictionary — all columns and their descriptions
variable_data = [
    # Demographics
    {'Variable Name': 'q1',  'Meaning': 'Age',                                'Tag': 'Demographics'},
    {'Variable Name': 'q2',  'Meaning': 'Sex',                                'Tag': 'Demographics'},
    {'Variable Name': 'q3',  'Meaning': 'Grade',                              'Tag': 'Demographics'},
    {'Variable Name': 'raceeth', 'Meaning': 'race and ethinicity combined',   'Tag': 'Demographics'},

    # Violence Exposure
    {'Variable Name': 'q15', 'Meaning': 'Threatened with weapon at School',   'Tag': 'Violence Exposure'},
    {'Variable Name': 'q16', 'Meaning': 'In a physical fight',                'Tag': 'Violence Exposure'},
    {'Variable Name': 'q17', 'Meaning': 'In a physical fight at school',      'Tag': 'Violence Exposure'},
    {'Variable Name': 'q18', 'Meaning': 'Seen physical violence',             'Tag': 'Violence Exposure'},

    # Sexual Violence, Violence Exposure
    {'Variable Name': 'q19', 'Meaning': 'Physically Forced to have sex',      'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q20', 'Meaning': 'Times Forced to do sexual things',   'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q21', 'Meaning': 'Forced by partner to do sexual things', 'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q22', 'Meaning': 'Physically hurt by partner',         'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q88', 'Meaning': 'Sexual abuse by older adult',        'Tag': 'Sexual Violence, Violence Exposure'},

    # Parental Abuse, Violence Exposure
    {'Variable Name': 'q90', 'Meaning': 'Parent/adult physical abuse (lifetime)', 'Tag': 'Parental Abuse, Violence Exposure'},

    # Bullying
    {'Variable Name': 'q24',  'Meaning': 'Bullied at School',                  'Tag': 'Bullying'},
    {'Variable Name': 'q25',  'Meaning': 'Electronically(Cyber?) bullied',     'Tag': 'Bullying'},
    {'Variable Name': 'q105', 'Meaning': 'Unfairly disciplined at school (12 months)', 'Tag': 'Bullying'},

    # Tobacco/Nicotine Use
    {'Variable Name': 'q31', 'Meaning': 'Ever smoked cigarette',              'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q32', 'Meaning': 'Age first smoked cigarette',         'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q33', 'Meaning': 'Days smoked cigarettes (30 days)',   'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q34', 'Meaning': 'Cigs/day on days smoked',           'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q35', 'Meaning': 'Ever used electronic vapor product', 'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q36', 'Meaning': 'Days used vapor product (30 days)', 'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q38', 'Meaning': 'Days used chewing tobacco (30 days)','Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q39', 'Meaning': 'Days smoked cigars (30 days)',       'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q40', 'Meaning': 'Tried to quit all tobacco (12 months)', 'Tag': 'Tobacco/Nicotine Use'},

    # Alcohol Use
    {'Variable Name': 'q41', 'Meaning': 'Age first drank alcohol',           'Tag': 'Alcohol Use'},
    {'Variable Name': 'q42', 'Meaning': 'Days drank alcohol (30 days)',      'Tag': 'Alcohol Use'},
    {'Variable Name': 'q43', 'Meaning': 'Days had 4+/5+ drinks in row (30 days)', 'Tag': 'Alcohol Use'},
    {'Variable Name': 'q44', 'Meaning': 'Max drinks in a row (30 days)',     'Tag': 'Alcohol Use'},
    {'Variable Name': 'q60', 'Meaning': 'Used alcohol/drugs before last sex','Tag': 'Alcohol Use'},

    # Other Drugs
    {'Variable Name': 'q46', 'Meaning': 'Times used marijuana (lifetime)',   'Tag': 'Other Drugs'},
    {'Variable Name': 'q47', 'Meaning': 'Age first tried marijuana',         'Tag': 'Other Drugs'},
    {'Variable Name': 'q48', 'Meaning': 'Times used marijuana (30 days)',    'Tag': 'Other Drugs'},
    {'Variable Name': 'q49', 'Meaning': 'Times misused pain medicine during lifetime', 'Tag': 'Other Drugs'},
    {'Variable Name': 'q92', 'Meaning': 'Times misused pain medicine during past 30 days', 'Tag': 'Other Drugs'},
    {'Variable Name': 'q50', 'Meaning': 'Times used cocaine',                'Tag': 'Other Drugs'},
    {'Variable Name': 'q51', 'Meaning': 'Times used inhalants',              'Tag': 'Other Drugs'},
    {'Variable Name': 'q52', 'Meaning': 'Times used heroin',                 'Tag': 'Other Drugs'},
    {'Variable Name': 'q53', 'Meaning': 'Times used methamphetamines',       'Tag': 'Other Drugs'},
    {'Variable Name': 'q54', 'Meaning': 'Times used ecstasy',                'Tag': 'Other Drugs'},
    {'Variable Name': 'q55', 'Meaning': 'Times injected illegal drugs',      'Tag': 'Other Drugs'},
    {'Variable Name': 'q93', 'Meaning': 'Times used hallucinogens (lifetime)','Tag': 'Other Drugs'},

    # ACEs
    {'Variable Name': 'q9',  'Meaning': 'Times rode with drunk driver (30 days)', 'Tag': 'ACEs'},
    {'Variable Name': 'q86', 'Meaning': 'Slept usually where (30 days)',     'Tag': 'Sleep, ACEs'},

    # Parental, ACEs
    {'Variable Name': 'q89', 'Meaning': 'Parent/adult insulted/put down (lifetime)', 'Tag': 'Parental, ACEs'},
    {'Variable Name': 'q91', 'Meaning': 'Parents/adults fought each other (lifetime)', 'Tag': 'Parental, ACEs'},

    # Parental Addiction/Mental Health/Separation, ACEs
    {'Variable Name': 'q100', 'Meaning': 'Lived with addict parent/guardian',  'Tag': 'Parental Addiction, ACEs'},
    {'Variable Name': 'q101', 'Meaning': 'Lived with mentally ill parent/guardian', 'Tag': 'Parental Mental Health, ACEs'},
    {'Variable Name': 'q102', 'Meaning': 'Separated from parent because of incarceration', 'Tag': 'Parental Separation, ACEs'},

    # Protective/Social Support Factors
    {'Variable Name': 'q99',  'Meaning': 'Adult met basic needs (lifetime)',  'Tag': 'Parental, Support'},
    {'Variable Name': 'q104', 'Meaning': 'An Adult knows whereabouts',       'Tag': 'Supervision, Support'},
    {'Variable Name': 'q87',  'Meaning': 'Grades in school (12 months)',     'Tag': 'Academic, Support'},
    {'Variable Name': 'q103', 'Meaning': 'Feel close to people at school',   'Tag': 'School, Support'},
    {'Variable Name': 'q83',  'Meaning': 'Last dental visit',                'Tag': 'Medical, Support'},

    # Sleep
    {'Variable Name': 'q85', 'Meaning': 'Sleep hours (school night)',        'Tag': 'Sleep'},

    # Physical Activity
    {'Variable Name': 'q76', 'Meaning': 'Days active 60+ min (7 days)',      'Tag': 'Physical Activity'},
    {'Variable Name': 'q77', 'Meaning': 'PE class days (week)',              'Tag': 'Physical Activity'},
    {'Variable Name': 'q78', 'Meaning': 'Sports teams played (12 months)',   'Tag': 'Physical Activity'},

    # Diet
    {'Variable Name': 'q68', 'Meaning': 'Drank 100% fruit juice (7 days)',  'Tag': 'Diet'},
    {'Variable Name': 'q69', 'Meaning': 'Ate fruit (7 days)',               'Tag': 'Diet'},
    {'Variable Name': 'q70', 'Meaning': 'Ate green salad (7 days)',         'Tag': 'Diet'},
    {'Variable Name': 'q71', 'Meaning': 'Ate potatoes (7 days)',            'Tag': 'Diet'},
    {'Variable Name': 'q72', 'Meaning': 'Ate carrots (7 days)',             'Tag': 'Diet'},
    {'Variable Name': 'q73', 'Meaning': 'Ate vegetables (7 days)',          'Tag': 'Diet'},
    {'Variable Name': 'q74', 'Meaning': 'Drank soda/pop (7 days)',          'Tag': 'Diet'},
    {'Variable Name': 'q75', 'Meaning': 'Days ate breakfast (7 days)',      'Tag': 'Diet'},

    # Social
    {'Variable Name': 'q80', 'Meaning': 'Social media use frequency',       'Tag': 'Social'},

    # Race, Bullying
    {'Variable Name': 'q23', 'Meaning': 'Treated badly because of race/ethnicity', 'Tag': 'Race, Bullying'},
]

variable_info_df = pd.DataFrame(variable_data)
print(f"Variable dictionary: {len(variable_data)} variables documented")
variable_info_df.head()

Variable dictionary: 69 variables documented


,Variable Name,Meaning,Tag
0,q1,Age,Demographics
1,q2,Sex,Demographics
2,q3,Grade,Demographics
3,raceeth,race and ethinicity combined,Demographics
4,q15,Threatened with weapon at School,Violence Exposure


In [7]:
# Utility: weighted prevalence across multiple outcome categories
def weighted_prevalence_multi(df, group_col, group_val, outcome_col, weight_col, outcome_vals=None):
    """
    Calculate weighted distribution for multiple outcome categories.

    Returns:
        dict: outcome_val -> {"weighted_count": ..., "percentage": ...}
    """
    valid_idx = (
        df[group_col].notna() &
        df[outcome_col].notna() &
        df[weight_col].notna()
    )
    df_valid = df.loc[valid_idx]
    group_df = df_valid[df_valid[group_col] == group_val]

    # total weighted denominator for a group
    denominator = np.sum(group_df[weight_col])

    if outcome_vals is None:
        outcome_vals = sorted(df_valid[outcome_col].unique())

    prevalence_dict = {}
    for val in outcome_vals:
        numerator = np.sum(group_df[weight_col] * (group_df[outcome_col] == val))
        prevalence = (numerator / denominator) if denominator > 0 else 0
        prevalence_dict[val] = {
            "weighted_count": numerator,
            "percentage": prevalence
        }

    return prevalence_dict

## 2. Data Preprocessing

Sequentially builds all derived variables needed for modeling and visualization.

### 2.1 Race/Ethnicity Mapping

In [8]:
# Race/ethnicity reference table from YRBS user guide
raw_raceeth_mapping_data = [
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "B", "Race": "A",                   "Value": 1, "Label": "American Indian/Alaskan Native"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "B", "Race": "B",                   "Value": 2, "Label": "Asian"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "B", "Race": "C",                   "Value": 3, "Label": "Black or African American"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "B", "Race": "D",                   "Value": 4, "Label": "Native Hawaiian or Other Pacific Islander"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "B", "Race": "E",                   "Value": 5, "Label": "White"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "A", "Race": "Missing",             "Value": 6, "Label": "Hispanic/Latino"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "A", "Race": "1 or more responses", "Value": 7, "Label": "Multiple– Hispanic/Latino"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "B", "Race": "2 or more responses", "Value": 8, "Label": "Multiple– Non-Hispanic/Latino"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "B", "Race": "Missing",             "Value": pd.NA, "Label": "Missing"},
    {"Category": "Ethnicity Race Raceeth", "Ethnicity": "Missing", "Race": "Missing or Any response", "Value": pd.NA, "Label": "Missing"},
]
raceeth_mapping_df = pd.DataFrame(raw_raceeth_mapping_data)
raceeth_mapping_df

,Category,Ethnicity,Race,Value,Label
0,Ethnicity Race Raceeth,B,A,1,American Indian/Alaskan Native
1,Ethnicity Race Raceeth,B,B,2,Asian
2,Ethnicity Race Raceeth,B,C,3,Black or African American
3,Ethnicity Race Raceeth,B,D,4,Native Hawaiian or Other Pacific Islander
4,Ethnicity Race Raceeth,B,E,5,White
5,Ethnicity Race Raceeth,A,Missing,6,Hispanic/Latino
6,Ethnicity Race Raceeth,A,1 or more responses,7,Multiple– Hispanic/Latino
7,Ethnicity Race Raceeth,B,2 or more responses,8,Multiple– Non-Hispanic/Latino
8,Ethnicity Race Raceeth,B,Missing,<NA>,Missing
9,Ethnicity Race Raceeth,Missing,Missing or Any response,<NA>,Missing


In [9]:
raceeth = raw_data["raceeth"]
raceeth_mapping_dict = {
    1: "American Indian/Alaskan Native",
    2: "Asian",
    3: "Black or African American",
    4: "Native Hawaiian or Other Pacific Islander",
    5: "White",
    6: "Hispanic/Latino",
    7: "Multiple – Hispanic/Latino",
    8: "Multiple – Non-Hispanic/Latino",
}

raceeth_labels = raceeth.map(raceeth_mapping_dict)
raw_data['raceeth_label'] = raceeth_labels
raw_data['raceeth'] = raw_data['raceeth'].astype("float32")

print("Descriptive Summary Statistics for 'raceeth':")
print(raceeth.describe())
freq_table = raceeth_labels.value_counts(dropna=False)
print("\nFrequency Table (including NaNs):")
print(freq_table)

raceeth_no_nan = raceeth_labels.dropna()
fig = px.bar(
    raceeth_no_nan.value_counts().sort_values(),
    orientation='h',
    labels={'value': 'Count', 'index': 'Race/Ethnicity'},
    title="Distribution of Race/Ethnicity ('raceeth')",
    text_auto=True
)
fig.update_traces(hovertemplate='Race/Ethnicity: %{y}<br>Count: %{x}')
fig.show()

Descriptive Summary Statistics for 'raceeth':
count    19733.000000
mean         5.010845
std          1.832258
min          1.000000
25%          5.000000
50%          5.000000
75%          6.000000
max          8.000000
Name: raceeth, dtype: float64

Frequency Table (including NaNs):
raceeth
White                                        9700
Multiple – Hispanic/Latino                   2786
Multiple – Non-Hispanic/Latino               1814
Black or African American                    1791
American Indian/Alaskan Native               1334
Hispanic/Latino                              1208
Asian                                         995
NaN                                           370
Native Hawaiian or Other Pacific Islander     105
Name: count, dtype: int64


### 2.2 Sexual Contact Variable

In [23]:
print("Unique values in Sex:", raw_data['q2'].unique())
print("Unique values in q63:", raw_data['q63'].unique())

Unique values in Sex: [ 1.  2. nan]
Unique values in q63: [ 1.  4.  3.  2. nan]


In [26]:
# Recode q63 (who had sexual contact with) relative to respondent sex
# This harmonises same-sex vs opposite-sex contact across male/female respondents
def assign_sexcontact(row):
    sex = row['q2']
    q63 = row['q63']

    if np.isnan(sex) or np.isnan(q63):
        return None
    sex = int(sex)
    q63 = int(q63)

    if sex == 1:  # Female
        if q63 == 1: return 4  # No sexual contact
        elif q63 == 2: return 2  # Female contact (same sex)
        elif q63 == 3: return 1  # Male contact (opposite sex)
        elif q63 == 4: return 3  # Both
    elif sex == 2:  # Male
        if q63 == 1: return 4
        elif q63 == 2: return 1
        elif q63 == 3: return 2
        elif q63 == 4: return 3
    return None

raw_data['sexcontact'] = raw_data.apply(assign_sexcontact, axis=1)

sexcontact_labels = {
    1: 'Opposite sex only',
    2: 'Same sex only',
    3: 'Both sexes',
    4: 'No sexual contact'
}
sex_labels = {1: 'Female', 2: 'Male'}

raw_data['sexcontact_label'] = raw_data['sexcontact'].map(sexcontact_labels)
raw_data['sex_label'] = raw_data['q2'].map(sex_labels)

In [27]:
# Distribution of Sexual Contact by Sex
plot_data = raw_data.dropna(subset=['sexcontact_label', 'sex_label'])

fig = px.histogram(
    plot_data,
    y='sexcontact_label',
    color='sex_label',
    text_auto=True,
    barnorm=None,
    category_orders={"sexcontact_label": ['No sexual contact', 'Opposite sex only', 'Same sex only', 'Both sexes']},
    labels={"sexcontact_label": "Sexual Contact Category", "count": "Count", "sex_label": "Sex"},
    title="Distribution of Sexual Contact by Sex"
)
fig.update_traces(hovertemplate='%{y}<br>Count: %{x}')
fig.show()

### 2.3 Sexual Identity Variable

In [28]:
# Rename q64 to sex_iden for clarity
raw_data = raw_data.rename(columns={'q64': 'sex_iden'})

sex_iden_labels = {
    1: 'Heterosexual (straight)',
    2: 'Gay or lesbian',
    3: 'Bisexual',
    4: 'Other way (self-described)',
    5: 'Questioning/not sure',
    6: 'Don’t understand the question',
}

raw_data['sex_iden_label'] = raw_data['sex_iden'].map(sex_iden_labels)

print("Descriptive Summary Statistics for 'sex_iden_label':")
print(raw_data['sex_iden_label'].describe())
freq_table = raw_data['sex_iden_label'].value_counts(dropna=False)
print("\nFrequency Table (including NaNs):")
print(freq_table)

plot_data = raw_data.dropna(subset=['sex_iden_label', 'sex_iden', 'sex_label'])

fig = px.histogram(
    plot_data,
    y='sex_iden_label',
    color='sex_label',
    text_auto=True,
    barnorm=None,
    category_orders={"sex_iden_label": [
        'Heterosexual (straight)', 'Gay or lesbian', 'Bisexual',
        'Other way (self-described)', 'Questioning/not sure', 'Don’t understand the question'
    ]},
    labels={"sex_iden_label": "Sexual Identity", "count": "Count", "sex_label": "Sex"},
    title="Distribution of Sexual Identity by Sex"
)
fig.update_traces(hovertemplate='%{y}<br>Count: %{x}')
fig.show()

Descriptive Summary Statistics for 'sex_iden_label':
count                       18100
unique                          6
top       Heterosexual (straight)
freq                        13289
Name: sex_iden_label, dtype: object

Frequency Table (including NaNs):
sex_iden_label
Heterosexual (straight)          13289
Bisexual                          2053
NaN                               2003
Questioning/not sure               850
Other way (self-described)         760
Gay or lesbian                     683
Don’t understand the question      465
Name: count, dtype: int64


In [29]:
# Map to binary minority/majority — includes questioning as minority
# (consistent with minority stress framework)
label_map = {
    'Heterosexual (straight)': 'Majority',
    'Gay or lesbian': 'Minority',
    'Bisexual': 'Minority',
    'Other way (self-described)': 'Minority',
    'Questioning/not sure': 'Minority',
    'Don’t understand the question': 'Majority'
}

numeric_map = {
    'Majority': 0,
    'Minority': 1,
    None: -1  # -1 for missing
}

raw_data['sex_iden_minority_label'] = raw_data['sex_iden_label'].map(label_map)
raw_data['sex_iden_minority'] = raw_data['sex_iden_minority_label'].map(numeric_map)

print(raw_data[['sex_iden_label', 'sex_iden_minority_label', 'sex_iden_minority']].head(10))
print("\nFrequency Table (including NaNs):")
print(raw_data['sex_iden_minority_label'].value_counts(dropna=False))
print(raw_data['sex_iden_minority'].value_counts(dropna=False))

plot_data = raw_data.dropna(subset=['sex_iden_minority_label', 'sex_label'])
fig = px.histogram(
    plot_data,
    y='sex_iden_minority_label',
    color='sex_label',
    text_auto=True,
    barnorm=None,
    category_orders={"sex_iden_minority_label": ['Majority', 'Minority']},
    labels={"sex_iden_minority_label": "Sexual Minority Status", "count": "Count", "sex_label": "Sex"},
    title="Distribution of Sexual Minority by Sex"
)
fig.update_traces(hovertemplate='%{y}<br>Count: %{x}')
fig.show()

            sex_iden_label sex_iden_minority_label  sex_iden_minority
0           Gay or lesbian                Minority                1.0
1  Heterosexual (straight)                Majority                0.0
2                 Bisexual                Minority                1.0
3  Heterosexual (straight)                Majority                0.0
4  Heterosexual (straight)                Majority                0.0
5  Heterosexual (straight)                Majority                0.0
6  Heterosexual (straight)                Majority                0.0
7                 Bisexual                Minority                1.0
8                 Bisexual                Minority                1.0
9                 Bisexual                Minority                1.0

Frequency Table (including NaNs):
sex_iden_minority_label
Majority    13754
Minority     4346
NaN          2003
Name: count, dtype: int64
sex_iden_minority
0.0    13754
1.0     4346
NaN     2003
Name: count, dtype: int64


### 2.4 Gender Minority Variable and Combined SGM Flag

In [30]:
# q65: transgender identity -> binary gender_minority
gender_minority_map = {
    1: 0,  # Not transgender (Majority)
    2: 1,  # Yes, transgender (Minority)
    3: 1,  # Not sure (Minority, included for broader minority)
    4: 0   # Don't understand (unknown group included in majority)
}
raw_data['gender_minority'] = raw_data['q65'].map(gender_minority_map)

# Combine sexual and gender minority into one SGM flag
def combine_sexual_gender_minority(row):
    if row['sex_iden_minority'] == 1 or row['gender_minority'] == 1:
        return 1         # Sexual or gender minority
    elif row['sex_iden_minority'] == 0 and row['gender_minority'] == 0:
        return 0         # Both majority
    else:
        return -1        # Unknown

raw_data['sex_iden_gender_minority'] = raw_data.apply(combine_sexual_gender_minority, axis=1)

combine_sexual_gender_minority_labels = {0: 'Majority', 1: 'Sexual or Gender Minority', -1: np.nan}
raw_data['sex_iden_gender_minority_label'] = raw_data['sex_iden_gender_minority'].map(combine_sexual_gender_minority_labels)

print(raw_data['sex_iden_gender_minority_label'].value_counts(dropna=False))

sex_iden_gender_minority_label
Majority                     13545
Sexual or Gender Minority     4451
NaN                           2107
Name: count, dtype: int64


### 2.5 Mental Health Composite Score

In [31]:
# Step 1: PCA-based factor score to derive weights
def create_factor_based_score_final(data):
    mh_matrix = pd.DataFrame()

    mh_matrix['sad']               = (data['q26'] == 1).astype(int)
    mh_matrix['ideation']          = (data['q27'] == 1).astype(int)
    mh_matrix['plan']              = (data['q28'] == 1).astype(int)
    mh_matrix['attempt']           = (data['q29'].isin([2, 3, 4, 5])).astype(int)
    mh_matrix['treatment']         = (data['q30'] == 2).astype(int)
    # q84: 1=Never, 5=Always; high-risk = 4-5 (Most/Always)
    mh_matrix['poor_mh_frequent']  = (data['q84'].isin([4, 5])).astype(int)
    # q106: 1=Has difficulty (risk), 2=No difficulty
    mh_matrix['cognitive_difficulty'] = (data['q106'] == 1).astype(int)

    # Drop missing — NA doesn't mean no symptoms
    mh_matrix = mh_matrix.dropna()

    scaler = StandardScaler()
    mh_scaled = scaler.fit_transform(mh_matrix)

    pca = PCA(n_components=1)
    factor_scores = pca.fit_transform(mh_scaled)

    print("Final Corrected Factor loadings:")
    for var, loading in zip(mh_matrix.columns, pca.components_[0]):
        print(f"  {var}: {loading:.3f}")
    print(f"\nVariance explained: {pca.explained_variance_ratio_[0]:.3f}")

    # Normalize to 0-10 scale
    factor_scores_normalized = (
        (factor_scores - factor_scores.min()) /
        (factor_scores.max() - factor_scores.min())
    ) * 10

    return factor_scores_normalized.flatten()

raw_data['mental_health_factor_score'] = create_factor_based_score_final(raw_data)

Final Corrected Factor loadings:
  sad: 0.404
  ideation: 0.465
  plan: 0.442
  attempt: 0.404
  treatment: 0.244
  poor_mh_frequent: 0.337
  cognitive_difficulty: 0.299

Variance explained: 0.426


In [32]:
# Step 2: Composite score using PCA-derived weights with clinical severity adjustments
def create_comprehensive_mental_health_composite(data):
    """
    Final mental health composite for risk stratification.
    Combines PCA-derived weights with clinical severity adjustments.
    Range: 0-10 (continuous risk score)
    """
    # PCA-derived weights
    pca_weights = {
        'sad':                 0.404,
        'ideation':            0.465,
        'plan':                0.442,
        'attempt':             0.404,
        'treatment':           0.244,
        'poor_mh_frequent':    0.337,
        'cognitive_difficulty': 0.299
    }

    score = 0
    score += pca_weights['sad']                 * (data['q26'] == 1).astype(int)
    score += pca_weights['ideation']            * (data['q27'] == 1).astype(int)
    score += pca_weights['plan']                * (data['q28'] == 1).astype(int)
    score += pca_weights['attempt']             * (data['q29'].isin([2, 3, 4, 5])).astype(int)
    score += pca_weights['treatment']           * (data['q30'] == 2).astype(int)
    score += pca_weights['poor_mh_frequent']    * (data['q84'].isin([4, 5])).astype(int)
    score += pca_weights['cognitive_difficulty']* (data['q106'] == 1).astype(int)

    # Normalize to 0-10 scale; max possible = sum of all weights = 2.595
    max_score = sum(pca_weights.values())
    normalized_score = (score / max_score) * 10

    return normalized_score

raw_data['mental_health_composite'] = create_comprehensive_mental_health_composite(raw_data)

print("Mental Health Composite Score Distribution:")
print(raw_data['mental_health_composite'].describe())

Mental Health Composite Score Distribution:
count    20103.000000
mean         2.059706
std          2.577626
min          0.000000
25%          0.000000
50%          1.152216
75%          3.348748
max         10.000000
Name: mental_health_composite, dtype: float64


In [33]:
# Step 3: Stratify into 3 risk categories
def stratify_mental_health_risk(score):
    """
    Stratification into 3 risk categories.

    Cutoffs based on:
    - Clinical severity (presence of suicide-related behaviors)
    - Data distribution (balanced classes)
    """
    if pd.isna(score):
        return np.nan
    if score <= 1.0:
        return 0  # Low Risk
    elif score <= 6.0:
        return 1  # Moderate Risk
    else:
        return 2  # High Risk

raw_data['mental_health_risk_category'] = raw_data['mental_health_composite'].apply(
    stratify_mental_health_risk
)

risk_labels = {0: 'Low Risk', 1: 'Moderate Risk', 2: 'High Risk'}
raw_data['mental_health_risk_label'] = raw_data['mental_health_risk_category'].map(risk_labels)

print("\nRisk Category Distribution")
print(raw_data['mental_health_risk_category'].value_counts().sort_index())
print("\nProportions:")
print(raw_data['mental_health_risk_category'].value_counts(normalize=True).sort_index())


Risk Category Distribution
mental_health_risk_category
0    9006
1    8865
2    2232
Name: count, dtype: int64

Proportions:
mental_health_risk_category
0    0.447993
1    0.440979
2    0.111028
Name: proportion, dtype: float64


In [34]:
# Validate risk categories: confirm gradient across symptom items
symptoms = {
    'q26':  {'name': 'sadness',        'condition': lambda df: (df['q26'] == 1)},
    'q27':  {'name': 'ideation',       'condition': lambda df: (df['q27'] == 1)},
    'q28':  {'name': 'plan',           'condition': lambda df: (df['q28'] == 1)},
    'q29':  {'name': 'attempt',        'condition': lambda df: df['q29'].isin([2, 3, 4, 5])},
    'q30':  {'name': 'med_treatment',  'condition': lambda df: (df['q30'] == 2)},
    'q84':  {'name': 'poor_mh',        'condition': lambda df: df['q84'].isin([4, 5])},
    'q106': {'name': 'cognitive_diff', 'condition': lambda df: (df['q106'] == 1)},
}

for var, info in symptoms.items():
    low_rate  = info['condition'](raw_data[raw_data['mental_health_risk_category'] == 0]).mean()
    mod_rate  = info['condition'](raw_data[raw_data['mental_health_risk_category'] == 1]).mean()
    high_rate = info['condition'](raw_data[raw_data['mental_health_risk_category'] == 2]).mean()
    gradient  = high_rate - low_rate
    print(f"{info['name']:15s} low: {low_rate:.1%}  mod: {mod_rate:.1%}  high: {high_rate:.1%}  gradient: {gradient:+.1%}")

sadness         low: 0.0%  mod: 66.6%  high: 98.8%  gradient: +98.8%
ideation        low: 0.0%  mod: 22.7%  high: 98.7%  gradient: +98.7%
plan            low: 0.0%  mod: 13.5%  high: 90.2%  gradient: +90.2%
attempt         low: 0.0%  mod: 5.4%  high: 67.9%  gradient: +67.9%
med_treatment   low: 0.0%  mod: 0.7%  high: 13.9%  gradient: +13.9%
poor_mh         low: 0.0%  mod: 36.8%  high: 69.4%  gradient: +69.4%
cognitive_diff  low: 0.0%  mod: 43.0%  high: 64.8%  gradient: +64.8%


### 2.6 Composite Variables (Visualization Only — Not Used in Modeling)

#### 2.6.1 Bullying Variable

In [35]:
# Composite bullying variable combining school, cyber, and disciplinary bullying.
# Note: q23 (race-based discrimination) is separated as a confounding variable
# to keep peer-bullying measure focused on LGBT-specific victimization.
# The confounding analysis is done in EDA below.

def combine_bullying(row):
    # q23: frequency, treat >2 (sometimes/most/always) as bullied
    experienced_discrimination = (row['q23'] in [3, 4, 5])
    bullied_at_school          = (row['q24'] == 1)
    cyber_bullied              = (row['q25'] == 1)
    unfairly_disciplined       = (row['q105'] == 1)

    if all(np.isnan([row['q23'], row['q24'], row['q25'], row['q105']])):
        return np.nan
    if experienced_discrimination or bullied_at_school or cyber_bullied or unfairly_disciplined:
        return 1  # Experienced bullying/discrimination
    else:
        return 0  # None experienced

def race_based_discrimination(row):
    """Separate measure for race-based discrimination — used as a control variable."""
    if pd.isna(row['q23']):
        return np.nan
    return 1 if row['q23'] in [3, 4, 5] else 0  # q23=0,1 is never and rarely

def peer_bullying_only(row):
    """Peer bullying without race-based discrimination — focused on LGBT-specific victimization."""
    bullied_at_school    = (row['q24'] == 1)
    cyber_bullied        = (row['q25'] == 1)
    unfairly_disciplined = (row['q105'] == 1)

    if all(pd.isna([row['q24'], row['q25'], row['q105']])):
        return np.nan
    if bullied_at_school or cyber_bullied or unfairly_disciplined:
        return 1
    else:
        return 0

raw_data['bullying_experience']  = raw_data.apply(combine_bullying, axis=1)
raw_data['race_discrimination']  = raw_data.apply(race_based_discrimination, axis=1)
raw_data['peer_bullying']        = raw_data.apply(peer_bullying_only, axis=1)

print("Bullying Experience:", raw_data['bullying_experience'].value_counts(dropna=False))
print("\nPeer bullying (LGBT-focused):", raw_data['peer_bullying'].value_counts(dropna=False))
print("\nRace-based discrimination (confounder):", raw_data['race_discrimination'].value_counts(dropna=False))

Bullying Experience: bullying_experience
0.0    12324
1.0     7749
NaN       30
Name: count, dtype: int64

Peer bullying (LGBT-focused): peer_bullying
0.0    13630
1.0     6414
NaN       59
Name: count, dtype: int64

Race-based discrimination (confounder): race_discrimination
0.0    15366
1.0     2920
NaN     1817
Name: count, dtype: int64


#### 2.6.2 Tobacco Intensity Variable

In [36]:
def tobacco_intensity(row):
    # No use: never tried cigarettes or vaping, and all day-use == 1 (zero days)
    no_use = (
        (row['q31'] == 2 or pd.isna(row['q31'])) and
        (row['q35'] == 2 or pd.isna(row['q35'])) and
        all((row[v] == 1 or pd.isna(row[v])) for v in ['q33', 'q36', 'q38', 'q39'])
    )
    if no_use:
        return 'No use'

    # Frequent/Heavy: any product used >= 4 days in past 30
    heavy_use = any(
        (row[v] is not np.nan) and (row[v] >= 4)
        for v in ['q33', 'q36', 'q38', 'q39']
    )
    if heavy_use:
        return 'Frequent/Heavy use'

    # Light/Experimental: ever tried or 2-5 day use
    lite_ever = (
        (row['q31'] == 1 or row['q35'] == 1) or
        any((row[v] in [2, 3])
            for v in ['q33', 'q36', 'q38', 'q39']
            if row[v] is not np.nan)
    )
    if lite_ever:
        return 'Light/Experimental use'

    return 'Missing'

raw_data['tobacco_use_intensity'] = raw_data.apply(tobacco_intensity, axis=1)
print(raw_data['tobacco_use_intensity'].value_counts(dropna=False))

tobacco_use_intensity
No use                    12530
Light/Experimental use     5224
Frequent/Heavy use         2349
Name: count, dtype: int64


#### 2.6.3 Alcohol Intensity Variable

In [37]:
def alcohol_intensity(row):
    # No use: never drank and no alcohol use or binge in past 30 days
    no_use = (
        (row['q41'] == 1 or pd.isna(row['q41'])) and
        (row['q42'] == 1 or pd.isna(row['q42'])) and
        (row['q43'] == 1 or pd.isna(row['q43'])) and
        (row['q44'] == 1 or pd.isna(row['q44']))
    )
    if no_use:
        return 'No use'

    # Frequent/Heavy: 6+ days drinking or binge, or high max drinks
    heavy = (
        (row['q42'] is not np.nan and row['q42'] >= 4) or
        (row['q43'] is not np.nan and row['q43'] >= 4) or
        (row['q44'] is not np.nan and row['q44'] >= 6)
    )
    if heavy:
        return 'Frequent/Heavy use'

    # Light/Experimental: has tried, low-frequency use
    light = (
        (row['q41'] != 1 and not pd.isna(row['q41'])) or
        (row['q42'] in [2, 3]) or
        (row['q43'] in [2, 3])
    )
    if light:
        return 'Light/Experimental use'

    return 'Missing'

raw_data['alcohol_use_intensity'] = raw_data.apply(alcohol_intensity, axis=1)
print(raw_data['alcohol_use_intensity'].value_counts(dropna=False))

alcohol_use_intensity
No use                    11089
Light/Experimental use     7577
Frequent/Heavy use         1432
Missing                       5
Name: count, dtype: int64


#### 2.6.4 Drug Intensity Variable

In [38]:
drug_vars = ['q46', 'q47', 'q48', 'q49', 'q50', 'q51', 'q52', 'q53', 'q54', 'q55', 'q92', 'q93']

def drug_intensity(row):
    no_use = all((row[v] == 1 or pd.isna(row[v])) for v in drug_vars)
    if no_use:
        return 'No use'

    # Frequent/Heavy: any substance used 10+ times (code >= 4)
    heavy_use = any((row[v] is not np.nan and row[v] >= 4) for v in drug_vars)
    if heavy_use:
        return 'Frequent/Heavy use'

    # Light/Experimental: at least one code 2-3
    light_exp = any((row[v] in [2, 3]) for v in drug_vars if row[v] is not np.nan)
    if light_exp:
        return 'Light/Experimental use'

    return 'Missing'

raw_data['drug_use_intensity'] = raw_data.apply(drug_intensity, axis=1)
print(raw_data['drug_use_intensity'].value_counts(dropna=False))

drug_use_intensity
No use                    12511
Frequent/Heavy use         6271
Light/Experimental use     1321
Name: count, dtype: int64


## 3. EDA Visualizations

### 3.1 Mental Health Risk by SGM Status

In [39]:
risk_labels_map = {0: 'Low Risk', 1: 'Moderate Risk', 2: 'High Risk'}
group_labels_map = {0: 'Majority (Straight)', 1: 'Minority (LGBT)'}
outcome_vals = [0, 1, 2]

maj_stats = weighted_prevalence_multi(
    raw_data, 'sex_iden_gender_minority', 0,
    'mental_health_risk_category', 'weight', outcome_vals=outcome_vals
)
min_stats = weighted_prevalence_multi(
    raw_data, 'sex_iden_gender_minority', 1,
    'mental_health_risk_category', 'weight', outcome_vals=outcome_vals
)

risk_cats  = [risk_labels_map[v] for v in outcome_vals]
maj_counts = [maj_stats[v]["weighted_count"] for v in outcome_vals]
min_counts = [min_stats[v]["weighted_count"] for v in outcome_vals]

fig_counts = go.Figure()
fig_counts.add_trace(go.Bar(
    name=group_labels_map[0], x=risk_cats, y=maj_counts,
    text=[f"{c:,.0f}" for c in maj_counts], textposition='outside', marker_color='#1f77b4'
))
fig_counts.add_trace(go.Bar(
    name=group_labels_map[1], x=risk_cats, y=min_counts,
    text=[f"{c:,.0f}" for c in min_counts], textposition='outside', marker_color='#d62728'
))
fig_counts.update_layout(
    barmode='group',
    xaxis_title="Mental Health Risk Category",
    yaxis_title="Weighted Count of Students",
    title="Weighted Counts of Mental Health Risk by Sexual and Gender Identity",
    legend_title="Group"
)
fig_counts.show()

maj_pcts = [maj_stats[v]["percentage"] * 100 for v in outcome_vals]
min_pcts = [min_stats[v]["percentage"] * 100 for v in outcome_vals]

fig_pct = go.Figure()
fig_pct.add_trace(go.Bar(
    name=group_labels_map[0], x=risk_cats, y=maj_pcts,
    text=[f"{p:.1f}%" for p in maj_pcts], textposition='outside', marker_color='#1f77b4'
))
fig_pct.add_trace(go.Bar(
    name=group_labels_map[1], x=risk_cats, y=min_pcts,
    text=[f"{p:.1f}%" for p in min_pcts], textposition='outside', marker_color='#d62728'
))
fig_pct.update_layout(
    barmode='group',
    xaxis_title="Mental Health Risk Category",
    yaxis_title="Weighted Percentage of Group (%)",
    title="Weighted Percentage in Each Mental Health Risk Category by Sexual and Gender Identity",
    legend_title="Group"
)
fig_pct.show()

### 3.2 Bullying Prevalence by SGM Status

In [40]:
print("Binary Outcome: Bullying Experience")

prev_minor_bullying = weighted_prevalence_multi(
    raw_data, 'sex_iden_gender_minority', 1, 'bullying_experience', 'weight', outcome_vals=[1]
)
prev_major_bullying = weighted_prevalence_multi(
    raw_data, 'sex_iden_gender_minority', 0, 'bullying_experience', 'weight', outcome_vals=[1]
)

pct_minority_bullying = prev_minor_bullying[1]["percentage"]
pct_majority_bullying = prev_major_bullying[1]["percentage"]

print(f"Weighted prevalence of bullying among sexual minority students: {pct_minority_bullying:.2%}")
print(f"Weighted prevalence of bullying among majority students: {pct_majority_bullying:.2%}")

groups      = ['Majority (Straight)', 'Minority (LGBT)']
percentages = [pct_majority_bullying * 100, pct_minority_bullying * 100]

fig1 = go.Figure(go.Bar(
    x=groups, y=percentages,
    text=[f"{v:.1f}%" for v in percentages],
    textposition='auto',
    marker_color=['#1f77b4', '#d62728']
))
fig1.update_layout(
    yaxis_title="Weighted % Reporting Bullying Experience",
    title="Weighted Prevalence of Bullying by Sexual and Gender Identity",
    xaxis_title="Group"
)
fig1.show()

Binary Outcome: Bullying Experience
Weighted prevalence of bullying among sexual minority students: 48.96%
Weighted prevalence of bullying among majority students: 34.39%


### 3.3 Tobacco Use Intensity by SGM Status

In [41]:
freq = (
    raw_data
    .dropna(subset=['tobacco_use_intensity', 'sex_iden_gender_minority_label', 'weight'])
    .groupby(['sex_iden_gender_minority_label', 'tobacco_use_intensity'])['weight']
    .sum()
    .reset_index()
)
freq['percent'] = freq.groupby('sex_iden_gender_minority_label')['weight'].transform(
    lambda x: 100 * x / x.sum()
)
print(freq)

fig = px.bar(
    freq, x='tobacco_use_intensity', y='percent', color='sex_iden_gender_minority_label',
    barmode='group',
    labels={
        "tobacco_use_intensity": "Tobacco/Nicotine Use Intensity",
        "percent": "Weighted %",
        "sex_iden_gender_minority_label": "Sexual & Gender Minority Status"
    },
    title="Weighted Percent: Tobacco/Nicotine Use Intensity by Sexual & Gender Minority Status",
    text='percent'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_title="Weighted Percent")
fig.show()

  sex_iden_gender_minority_label   tobacco_use_intensity     weight    percent
0                       Majority      Frequent/Heavy use  1252.2072   8.948103
1                       Majority  Light/Experimental use  3249.9237  23.223515
2                       Majority                  No use  9491.9769  67.828382
3      Sexual or Gender Minority      Frequent/Heavy use   696.9960  14.993456
4      Sexual or Gender Minority  Light/Experimental use  1368.9706  29.448663
5      Sexual or Gender Minority                  No use  2582.7015  55.557881


### 3.4 Alcohol Use Intensity by SGM Status

In [42]:
freq = (
    raw_data
    .dropna(subset=['alcohol_use_intensity', 'sex_iden_gender_minority_label', 'weight'])
    .groupby(['sex_iden_gender_minority_label', 'alcohol_use_intensity'])['weight']
    .sum()
    .reset_index()
)
freq['percent'] = freq.groupby('sex_iden_gender_minority_label')['weight'].transform(
    lambda x: 100 * x / x.sum()
)
print(freq)

fig = px.bar(
    freq, x='alcohol_use_intensity', y='percent', color='sex_iden_gender_minority_label',
    barmode='group',
    labels={
        "alcohol_use_intensity": "Alcohol Use Intensity",
        "percent": "Weighted %",
        "sex_iden_gender_minority_label": "Sexual & Gender Minority Status"
    },
    title="Weighted Percent: Alcohol Use Intensity by Sexual & Gender Minority Status",
    text='percent'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_title="Weighted Percent")
fig.show()

  sex_iden_gender_minority_label   alcohol_use_intensity     weight    percent
0                       Majority      Frequent/Heavy use   909.2874   6.497645
1                       Majority  Light/Experimental use  5186.0289  37.058661
2                       Majority                  No use  7898.7915  56.443695
3      Sexual or Gender Minority      Frequent/Heavy use   375.7096   8.082091
4      Sexual or Gender Minority  Light/Experimental use  1965.0701  42.271680
5      Sexual or Gender Minority                 Missing     0.3100   0.006669
6      Sexual or Gender Minority                  No use  2307.5784  49.639560


### 3.5 Drug Use Intensity by SGM Status

In [43]:
freq = (
    raw_data
    .dropna(subset=['drug_use_intensity', 'sex_iden_gender_minority_label', 'weight'])
    .groupby(['sex_iden_gender_minority_label', 'drug_use_intensity'])['weight']
    .sum()
    .reset_index()
)
freq['percent'] = freq.groupby('sex_iden_gender_minority_label')['weight'].transform(
    lambda x: 100 * x / x.sum()
)
print(freq)

fig = px.bar(
    freq, x='drug_use_intensity', y='percent', color='sex_iden_gender_minority_label',
    barmode='group',
    labels={
        "drug_use_intensity": "Drug Use Intensity",
        "percent": "Weighted %",
        "sex_iden_gender_minority_label": "Sexual & Gender Minority Status"
    },
    title="Weighted Percent: Drug Use Intensity by Sexual & Gender Minority Status",
    text='percent'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_title="Weighted Percent")
fig.show()

  sex_iden_gender_minority_label      drug_use_intensity     weight    percent
0                       Majority      Frequent/Heavy use  3698.2028  26.426857
1                       Majority  Light/Experimental use   796.4246   5.691142
2                       Majority                  No use  9499.4804  67.882001
3      Sexual or Gender Minority      Frequent/Heavy use  1912.7831  41.146906
4      Sexual or Gender Minority  Light/Experimental use   431.9215   9.291296
5      Sexual or Gender Minority                  No use  2303.9635  49.561798


### 3.6 Confounding Analysis: Race Discrimination and SGM Status

In [44]:
# Is race-based discrimination disproportionately reported by LGBT students?
# If yes, it is a confounder and should be controlled for separately.

crosstab = pd.crosstab(
    raw_data['sex_iden_gender_minority_label'],
    raw_data['race_discrimination'],
    normalize='index'
)
print("\nRace discrimination by LGBT status (row percentages):")
print(crosstab)

chi2, p, dof, expected = chi2_contingency(
    pd.crosstab(raw_data['sex_iden_gender_minority_label'], raw_data['race_discrimination'])
)
print(f"\nchi-square test: χ²={chi2:.2f}, p={p:.6f}")

if p < 0.05:
    print("Race discrimination IS associated with LGBT status (confounder present)")
else:
    print("Race discrimination NOT associated with LGBT status (not a confounder)")


Race discrimination by LGBT status (row percentages):
race_discrimination                  0.0       1.0
sex_iden_gender_minority_label                    
Majority                        0.857962  0.142038
Sexual or Gender Minority       0.796119  0.203881

chi-square test: χ²=92.54, p=0.000000
Race discrimination IS associated with LGBT status (confounder present)


In [45]:
# Does race discrimination predict mental health risk?
crosstab = pd.crosstab(
    raw_data['race_discrimination'],
    raw_data['mental_health_risk_category'],
    normalize='index'
)
print("\nRisk category distribution by race discrimination (row percentages):")
print(crosstab)
print()

crosstab_counts = pd.crosstab(
    raw_data['race_discrimination'],
    raw_data['mental_health_risk_category']
)
chi2, p_val, dof, expected = chi2_contingency(crosstab_counts)

print("chi-square test:")
if p_val < 0.001:
    print(f"  χ² = {chi2:.2f}, df = {dof}, p < 0.001")
else:
    print(f"  χ² = {chi2:.2f}, df = {dof}, p = {p_val:.4f}")

if p_val < 0.05:
    print("\n Race discrimination DOES predict mental health risk category (confounder confirmed)")
else:
    print("\n Race discrimination does NOT predict mental health risk category")

high_risk_no_disc  = crosstab.loc[0, 2]
high_risk_yes_disc = crosstab.loc[1, 2]
risk_ratio = high_risk_yes_disc / high_risk_no_disc

print(f"\nhigh-risk category prevalence:")
print(f"  no race discrimination:  {high_risk_no_disc:.1%}")
print(f"  yes race discrimination: {high_risk_yes_disc:.1%}")
print(f"  risk ratio: {risk_ratio:.2f}x")
print(f"  absolute difference: +{(high_risk_yes_disc - high_risk_no_disc)*100:.1f} percentage points")


Risk category distribution by race discrimination (row percentages):
mental_health_risk_category         0         1         2
race_discrimination                                      
0.0                          0.481387  0.422101  0.096512
1.0                          0.263356  0.526370  0.210274

chi-square test:
  χ² = 598.91, df = 2, p < 0.001

 Race discrimination DOES predict mental health risk category (confounder confirmed)

high-risk category prevalence:
  no race discrimination:  9.7%
  yes race discrimination: 21.0%
  risk ratio: 2.18x
  absolute difference: +11.4 percentage points


## 4. Feature Selection and Engineering

### 4.1 Feature Subsetting

In [48]:
vars_list = [
    'Age', 'Sex', 'sex_label', 'Grade', 'raceeth', 'raceeth_label',
    'sex_iden_gender_minority', 'sex_iden_gender_minority_label',

    'q15', 'q16', 'q17', 'q18',
    'q19', 'q20', 'q21', 'q22', 'q88', 'q90',
    'q24', 'q25', 'q105',
    'q31', 'q32', 'q33', 'q34', 'q35', 'q36', 'q37', 'q38', 'q39', 'q40',
    'q41', 'q42', 'q43', 'q44', 'q60',
    'q46', 'q47', 'q48', 'q49', 'q92', 'q50', 'q51', 'q52', 'q53', 'q54', 'q55', 'q93',
    'q9', 'q86', 'q89', 'q91', 'q100', 'q101', 'q102',
    'q99', 'q104', 'q87', 'q103', 'q83',
    'q85', 'q76', 'q77', 'q78',
    'q68', 'q69', 'q70', 'q71', 'q72', 'q73', 'q74', 'q75',
    'q80',
    'q23',
    'weight',
]

target_var = 'mental_health_risk_category'
weight_var = 'weight'
raw_data = raw_data.rename(columns={
    'q1': 'Age',
    'q2': 'Sex',
    'q3': 'Grade',
    'q6': 'Height_meters',
    'q7': 'Weight_kg'
})
df = raw_data[vars_list + [target_var, weight_var]].copy()
df.dropna(subset=[target_var], inplace=True)
print(f"Modeling dataset shape: {df.shape}")

Modeling dataset shape: (20103, 77)


### 4.2 Value Labels Dictionary

In [49]:
value_labels = {
    "sex_label": {1: 'Female', 2: 'Male'},

    "age_bin": {
        0: "Early adolescence (12-14)",
        1: "Middle adolescence (15-16)",
        2: "Late adolescence (17-18+)",
    },

    "grade_label": {
        1: "9th grade", 2: "10th grade", 3: "11th grade",
        4: "12th grade", 5: "Ungraded or other grade",
    },

    "sex_iden_gender_minority": {0: 'Majority', 1: 'Sexual or Gender Minority'},

    "q15_times_threatw_weapon_12m_bin": {0: "None", 1: "Once", 2: "Few (2-5 times)", 3: "Many (6+ times)"},
    "q16_times_in_phys_fight_12m_bin":  {0: "None", 1: "Once", 2: "Few (2-5 times)", 3: "Many (6+ times)"},
    "q17_times_fight_in_school_12m_bin":{0: "None", 1: "Once", 2: "Few (2-5 times)", 3: "Many (6+ times)"},
    "q18_seen_phy_viol_bin":            {0: "No",   1: "Yes"},
    "q19_forced_sex_bin":               {0: "No",   1: "Yes"},
    "q88_adult_forced_sex_bin":         {0: "No",   1: "Yes"},
    "q20_times_forced_sex_12m_bin":     {0: "None", 1: "Once", 2: "Few (2-5 times)", 3: "Many (6+ times)"},
    "q21_times_part_forced_sex_12m_bin":{0: "None", 1: "Once", 2: "Few (2-5 times)", 3: "Many (6+ times)"},
    "q22_times_part_hurt_12m_bin":      {0: "None", 1: "Once", 2: "Few (2-5 times)", 3: "Many (6+ times)"},
    "q89_times_insulted_byparent_l_bin":{0: "None / rarely", 1: "Sometimes", 2: "Often / always"},
    "q90_times_hurt_byadult_l_bin":     {0: "None / rarely", 1: "Sometimes", 2: "Often / always"},
    "q91_times_adults_phy_fought_30_bin":{0: "None / rarely",1: "Sometimes", 2: "Often / always"},
    "q23_times_treated_bad_atschool_race_bin": {0: "Never / rarely", 1: "Sometimes", 2: "Frequent"},
    "q23_treated_bad_atschool_race_ever":      {0: "Never", 1: "Ever"},
    "q24_bullied_atschool_12m_bin":     {0: "No",   1: "Yes"},
    "q25_cyber_bullied_12m_bin":        {0: "No",   1: "Yes"},
    "q105_unfairly_disciplined_12m_bin":{0: "No",   1: "Yes"},

    "q31_smoked_cig_bin":     {0: "Never", 1: "Ever smoked cigarettes"},
    "q32_age_1st_cig_bin":    {0: "Never", 1: "<13 years", 2: "13-14 years", 3: "15-16 years", 4: "17+ years"},
    "q33_days_smoked_cig_30_bin": {0: "None", 1: "Rare", 2: "Occasional", 3: "Frequent"},
    "q36_days_vaped_30_bin":  {0: "None",  1: "Rare", 2: "Occasional", 3: "Frequent"},
    "q38_days_tob_30_bin":    {0: "None",  1: "Rare", 2: "Occasional", 3: "Frequent"},
    "q39_days_smoked_cigars_30_bin": {0: "None", 1: "Rare", 2: "Occasional", 3: "Frequent"},
    "q34_no_of_cigs_pd_30_bin": {0: "None", 1: "Light", 2: "Moderate", 3: "Heavy"},
    "q35_ever_vaped_bin":     {0: "No",    1: "Yes"},
    "q40_tried_to_quit_12m_bin": {0: "Did not use", 1: "Tried to quit", 2: "Did not try to quit"},
    "q41_age_1st_drink_bin":  {0: "Never", 1: "<11 years", 2: "11-14 years", 3: "15+ years"},
    "q42_days_drank_alc_30_bin": {0: "None", 1: "Rare", 2: "Occasional", 3: "Frequent"},
    "q43_days_binge_alcdrink_30_bin": {0: "None", 1: "1-2 days", 2: "3-5 days", 3: "Frequent (6+ days)"},
    "q43_any_alcdrink_binge": {0: "No binge", 1: "Any binge drinking"},
    "q44_max_drinks_30_bin":  {0: "None", 1: "1-2 drinks", 2: "3-4 drinks", 3: "5-7 drinks", 4: "8+ drinks"},
    "q60_subs_before_lastsex_bin": {0: "No or never had sex", 1: "Yes, used alcohol/drugs before last sex"},
    "q46_times_used_marijuana_l_bin": {0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q48_times_used_marijuana_30_bin":{0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q49_times_pain_med_abuse_l_bin": {0: "Never", 1: "Experimented", 2: "Some misuse", 3: "Frequent misuse"},
    "q92_times_pain_med_abuse_30_bin":{0: "Never", 1: "Experimented", 2: "Some misuse", 3: "Frequent misuse"},
    "q50_times_used_cocaine_l_bin":   {0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q51_times_sniffed_aerosol_l_bin":{0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q52_times_used_heroin_l_bin":    {0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q53_times_used_methamph_l_bin":  {0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q54_times_used_ecstasy_l_bin":   {0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q93_times_used_hallucin_l_bin":  {0: "Never", 1: "Experimented", 2: "Some use", 3: "Frequent use"},
    "q55_times_inject_drugs_l_bin":   {0: "Never", 1: "1 time", 2: "2+ times"},
    "q47_age_1st_mari_bin":           {0: "Never", 1: "<11 years", 2: "11-14 years", 3: "15+ years"},

    "q9_times_rode_w_drunkdri_30_bin": {0: "None", 1: "Once", 2: "Few times", 3: "Many times"},
    "q9_rode_wdrunkdriver_any":         {0: "Never", 1: "Ever"},
    "q86_slept_where_30_bin":  {0: "Home", 1: "With others", 2: "Unstable / other setting"},
    "q86_stable_sleep":        {0: "Not stable (non-home)", 1: "Stable (home)"},
    "q100_lived_w_guardian_w_drugalc_issues_l_bin": {0: "No", 1: "Yes (guardian with alcohol/drug issues)"},
    "q101_lived_w_severely_mentally_ill_guardian_l_bin": {0: "No", 1: "Yes (guardian severely mentally ill)"},
    "q102_separated_from_guardian_l_bin": {0: "No", 1: "Yes (separated due to incarceration)"},
    "q99_basic_needs_met_l_bin": {0: "Inconsistent support", 1: "Sometimes", 2: "Consistent support"},
    "q99_high_support":          {0: "Low / inconsistent support", 1: "High support"},
    "q104_adult_knew_whereabout_bin": {0: "Inconsistent supervision", 1: "Sometimes", 2: "Consistent supervision"},
    "q87_grades_12m_bin":    {0: "High grades", 1: "Mid grades", 2: "Low / unknown grades"},
    "q87_high_grades":       {0: "Not high grades", 1: "High grades"},
    "q103_feel_close_to_ppl_atschool_bin": {0: "Agree", 1: "Not sure", 2: "Disagree"},
    "q83_last_dental_visit_bin": {0: "Recent (past year)", 1: "Within 2 years", 2: "Overdue / never / unknown"},
    "q83_recent_dental":         {0: "Not recent", 1: "Recent dental visit"},

    "q68_juice_freq_bin":    {0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q69_fruit_freq_bin":    {0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q70_salad_freq_bin":    {0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q71_potato_freq_bin":   {0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q72_carrot_freq_bin":   {0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q73_other_veg_freq_bin":{0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q74_soda_freq_bin":     {0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q75_breakfast_days_bin":{0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q76_days_phys_active_7_bin": {0: "Inactive (0 days)", 1: "1-2 days", 2: "3-4 days", 3: "5-6 days", 4: "7 days"},
    "q77_days_pe_class_7_bin":    {0: "None", 1: "1-2 days", 2: "3+ days"},
    "q78_no_sports_team_12m_bin": {0: "No teams", 1: "1 team", 2: "2 teams", 3: "3+ teams"},
    "q80_freq_soc_media_bin":     {0: "None", 1: "Low", 2: "Moderate", 3: "High"},
    "q85_hours_slept_bin":        {0: "Short sleep", 1: "Adequate sleep", 2: "Long sleep"},
}

def describe_code(var, code):
    if pd.isna(code):
        return "Missing"
    return value_labels.get(var, {}).get(code, f"{var}={code}")

### 4.3 Binning Functions

In [51]:
# Demographics
# Age: 0=Early, 1=Middle, 2=Late adolescence
def age_bin_numeric(age_code):
    if pd.isna(age_code): return np.nan
    age_code = int(age_code)
    if age_code in [1, 2, 3]: return 0   # Early Adolescence
    elif age_code in [4, 5]:  return 1   # Middle Adolescence
    elif age_code in [6, 7]:  return 2   # Late Adolescence
    else: return np.nan

df["age_bin"] = df["Age"].apply(age_bin_numeric).astype("float32")

df["grade"] = df["Grade"].where(df["Grade"].notna(), np.nan).astype("float32")
grade_label_map = value_labels["grade_label"]
df["grade_label"] = df["grade"].map(grade_label_map)

# Violence Exposure
# 0=None, 1=Once, 2=Few, 3=Many
def violence_freq_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:           return 0
    elif val == 2:         return 1
    elif val in [3, 4]:   return 2
    elif val in [5,6,7,8]:return 3
    else: return np.nan

df["q15_times_threatw_weapon_12m_bin"] = df["q15"].apply(violence_freq_bin).astype("float32")
df["q16_times_in_phys_fight_12m_bin"]  = df["q16"].apply(violence_freq_bin).astype("float32")
df["q17_times_fight_in_school_12m_bin"]= df["q17"].apply(violence_freq_bin).astype("float32")

# Seen physical violence: binary
df["q18_seen_phy_viol_bin"] = df["q18"].map({1: 1, 2: 0}).astype("float32")

# Sexual violence (lifetime): binary
df["q19_forced_sex_bin"]     = df["q19"].map({1: 1, 2: 0}).astype("float32")
df["q88_adult_forced_sex_bin"]= df["q88"].map({1: 1, 2: 0}).astype("float32")

# Forced sex frequency (12m): 0=None,1=Once,2=Few,3=Many
def forced_sex_freq_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1: return 0
    elif val == 2: return 1
    elif val in [3, 4]: return 2
    elif val == 5: return 3
    else: return np.nan

df["q20_times_forced_sex_12m_bin"] = df["q20"].apply(forced_sex_freq_bin).astype("float32")

# Dating violence frequency
def dating_violence_freq_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [1, 2]: return 0
    elif val == 3:    return 1
    elif val in [4, 5]: return 2
    elif val == 6:    return 3
    else: return np.nan

df["q21_times_part_forced_sex_12m_bin"] = df["q21"].apply(dating_violence_freq_bin).astype("float32")
df["q22_times_part_hurt_12m_bin"]        = df["q22"].apply(dating_violence_freq_bin).astype("float32")

# Parental abuse: 0=None/Rarely,1=Sometimes,2=Often/Always
def abuse_freq_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [1, 2]: return 0
    elif val == 3:    return 1
    elif val in [4, 5]: return 2
    else: return np.nan

df["q89_times_insulted_byparent_l_bin"]  = df["q89"].apply(abuse_freq_bin).astype("float32")
df["q90_times_hurt_byadult_l_bin"]       = df["q90"].apply(abuse_freq_bin).astype("float32")
df["q91_times_adults_phy_fought_30_bin"] = df["q91"].apply(abuse_freq_bin).astype("float32")

# Bullying
def race_bullying_freq_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [1, 2]: return 0
    elif val == 3:    return 1
    elif val in [4, 5]: return 2
    else: return np.nan

df["q23_times_treated_bad_atschool_race_bin"] = df["q23"].apply(race_bullying_freq_bin).astype("float32")
df["q23_treated_bad_atschool_race_ever"] = np.where(
    df["q23"].isna(), np.nan, (df["q23"].isin([2, 3, 4, 5])).astype("float32")
)

df["q24_bullied_atschool_12m_bin"]      = df["q24"].map({1: 1, 2: 0}).astype("float32")
df["q25_cyber_bullied_12m_bin"]         = df["q25"].map({1: 1, 2: 0}).astype("float32")
df["q105_unfairly_disciplined_12m_bin"] = df["q105"].map({1: 1, 2: 0}).astype("float32")

# Tobacco Use
df["q31_smoked_cig_bin"] = df["q31"].map({1: 1, 2: 0}).astype("float32")

def age_first_smoke_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1: return 0
    elif val in [2, 3, 4]: return 1
    elif val == 5: return 2
    elif val == 6: return 3
    elif val == 7: return 4
    else: return np.nan

df["q32_age_1st_cig_bin"] = df["q32"].apply(age_first_smoke_bin).astype("float32")

def days_used_freq_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5]: return 2
    elif val in [6,7]: return 3
    else: return np.nan

df["q33_days_smoked_cig_30_bin"]   = df["q33"].apply(days_used_freq_bin).astype("float32")
df["q36_days_vaped_30_bin"]        = df["q36"].apply(days_used_freq_bin).astype("float32")
df["q38_days_tob_30_bin"]          = df["q38"].apply(days_used_freq_bin).astype("float32")
df["q39_days_smoked_cigars_30_bin"]= df["q39"].apply(days_used_freq_bin).astype("float32")

def cigs_per_day_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5]: return 2
    elif val in [6,7]: return 3
    else: return np.nan

df["q34_no_of_cigs_pd_30_bin"] = df["q34"].apply(cigs_per_day_bin).astype("float32")
df["q35_ever_vaped_bin"]       = df["q35"].map({1: 1, 2: 0}).astype("float32")

def tried_quit_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1: return 0
    elif val == 2: return 1
    elif val == 3: return 2
    else: return np.nan

df["q40_tried_to_quit_12m_bin"] = df["q40"].apply(tried_quit_bin).astype("float32")

# Alcohol Use
def age_first_drink_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5]: return 2
    elif val in [6,7]: return 3
    else: return np.nan

df["q41_age_1st_drink_bin"] = df["q41"].apply(age_first_drink_bin).astype("float32")

def alcohol_days_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5]: return 2
    elif val in [6,7]: return 3
    else: return np.nan

df["q42_days_drank_alc_30_bin"] = df["q42"].apply(alcohol_days_bin).astype("float32")

def binge_days_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val == 4:     return 2
    elif val in [5,6,7]: return 3
    else: return np.nan

df["q43_days_binge_alcdrink_30_bin"] = df["q43"].apply(binge_days_bin).astype("float32")
df["q43_any_alcdrink_binge"] = np.where(
    df["q43"].isna(), np.nan, (df["q43"] > 1).astype("float32")
)

def max_drinks_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val == 2:     return 1
    elif val in [3,4]: return 2
    elif val in [5,6]: return 3
    elif val in [7,8]: return 4
    else: return np.nan

df["q44_max_drinks_30_bin"] = df["q44"].apply(max_drinks_bin).astype("float32")

def substance_before_sex_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    return 1.0 if val == 2 else 0.0

df["q60_subs_before_lastsex_bin"] = df["q60"].apply(substance_before_sex_bin).astype("float32")

# Drug Use
def drug_use_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val == 2:     return 1
    elif val in [3,4]: return 2
    elif val in [5,6,7]: return 3
    else: return np.nan

def inject_drug_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1: return 0
    elif val == 2: return 1
    elif val == 3: return 2
    else: return np.nan

df["q46_times_used_marijuana_l_bin"]  = df["q46"].apply(drug_use_bin).astype("float32")
df["q48_times_used_marijuana_30_bin"] = df["q48"].apply(drug_use_bin).astype("float32")
df["q49_times_pain_med_abuse_l_bin"]  = df["q49"].apply(drug_use_bin).astype("float32")
df["q92_times_pain_med_abuse_30_bin"] = df["q92"].apply(drug_use_bin).astype("float32")
df["q50_times_used_cocaine_l_bin"]    = df["q50"].apply(drug_use_bin).astype("float32")
df["q51_times_sniffed_aerosol_l_bin"] = df["q51"].apply(drug_use_bin).astype("float32")
df["q52_times_used_heroin_l_bin"]     = df["q52"].apply(drug_use_bin).astype("float32")
df["q53_times_used_methamph_l_bin"]   = df["q53"].apply(drug_use_bin).astype("float32")
df["q54_times_used_ecstasy_l_bin"]    = df["q54"].apply(drug_use_bin).astype("float32")
df["q55_times_inject_drugs_l_bin"]    = df["q55"].apply(inject_drug_bin).astype("float32")
df["q93_times_used_hallucin_l_bin"]   = df["q93"].apply(drug_use_bin).astype("float32")

def age_first_marijuana_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5]: return 2
    elif val in [6,7]: return 3
    else: return np.nan

df["q47_age_1st_mari_bin"] = df["q47"].apply(age_first_marijuana_bin).astype("float32")

# ACEs
def drunk_driver_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1: return 0
    elif val == 2: return 1
    elif val in [3,4]: return 2
    elif val == 5: return 3
    else: return np.nan

df["q9_times_rode_w_drunkdri_30_bin"] = df["q9"].apply(drunk_driver_bin).astype("float32")
df["q9_rode_wdrunkdriver_any"] = np.where(
    df["q9"].isna(), np.nan, (df["q9"] > 1).astype("float32")
)

def sleep_location_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:          return 0
    elif val == 2:        return 1
    elif val in [3,4,5,6,7]: return 2
    else: return np.nan

df["q86_slept_where_30_bin"] = df["q86"].apply(sleep_location_bin).astype("float32")
df["q86_stable_sleep"] = np.where(
    df["q86"].isna(), np.nan, (df["q86"] == 1).astype("float32")
)

df["q100_lived_w_guardian_w_drugalc_issues_l_bin"]      = df["q100"].map({1: 1, 2: 0}).astype("float32")
df["q101_lived_w_severely_mentally_ill_guardian_l_bin"] = df["q101"].map({1: 1, 2: 0}).astype("float32")
df["q102_separated_from_guardian_l_bin"]                = df["q102"].map({1: 1, 2: 0}).astype("float32")

# Protective / Social Support
def support_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [1,2]: return 0
    elif val == 3:   return 1
    elif val in [4,5]: return 2
    else: return np.nan

df["q99_basic_needs_met_l_bin"] = df["q99"].apply(support_bin).astype("float32")
df["q99_high_support"] = np.where(
    df["q99"].isna(), np.nan, (df["q99"].isin([4, 5])).astype("float32")
)
df["q104_adult_knew_whereabout_bin"] = df["q104"].apply(support_bin).astype("float32")

def grades_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [1,2]: return 0
    elif val in [3,4]: return 1
    elif val in [5,6,7]: return 2
    else: return np.nan

df["q87_grades_12m_bin"] = df["q87"].apply(grades_bin).astype("float32")
df["q87_high_grades"] = np.where(
    df["q87"].isna(), np.nan, (df["q87"].isin([1, 2])).astype("float32")
)

def connection_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [1,2]: return 0
    elif val == 3:   return 1
    elif val in [4,5]: return 2
    else: return np.nan

df["q103_feel_close_to_ppl_atschool_bin"] = df["q103"].apply(connection_bin).astype("float32")

def dental_visit_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val == 2:     return 1
    elif val in [3,4,5]: return 2
    else: return np.nan

df["q83_last_dental_visit_bin"] = df["q83"].apply(dental_visit_bin).astype("float32")
df["q83_recent_dental"] = np.where(
    df["q83"].isna(), np.nan, (df["q83"] == 1).astype("float32")
)

#  Diet, Physical Activity, Social Media, Sleep
def diet_freq_bin_numeric(val, seven_is_all=False):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5]: return 2
    else:
        if seven_is_all:
            if val in [6, 7, 8]: return 3
        else:
            if val in [6, 7]:    return 3
    return np.nan

df["q68_juice_freq_bin"]  = df["q68"].apply(lambda x: diet_freq_bin_numeric(x)).astype("float32")
df["q69_fruit_freq_bin"]  = df["q69"].apply(lambda x: diet_freq_bin_numeric(x)).astype("float32")
df["q70_salad_freq_bin"]  = df["q70"].apply(lambda x: diet_freq_bin_numeric(x)).astype("float32")
df["q71_potato_freq_bin"] = df["q71"].apply(lambda x: diet_freq_bin_numeric(x)).astype("float32")
df["q72_carrot_freq_bin"] = df["q72"].apply(lambda x: diet_freq_bin_numeric(x)).astype("float32")
df["q73_other_veg_freq_bin"]= df["q73"].apply(lambda x: diet_freq_bin_numeric(x)).astype("float32")
df["q74_soda_freq_bin"]   = df["q74"].apply(lambda x: diet_freq_bin_numeric(x)).astype("float32")
# Breakfast: code 8 = all 7 days
df["q75_breakfast_days_bin"] = df["q75"].apply(
    lambda x: diet_freq_bin_numeric(x, seven_is_all=True)
).astype("float32")

def pa_days_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5]: return 2
    elif val in [6,7]: return 3
    elif val == 8:     return 4
    else: return np.nan

df["q76_days_phys_active_7_bin"] = df["q76"].apply(pa_days_bin).astype("float32")

def pe_days_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:       return 0
    elif val in [2,3]: return 1
    elif val in [4,5,6]: return 2
    else: return np.nan

df["q77_days_pe_class_7_bin"] = df["q77"].apply(pe_days_bin).astype("float32")

def sports_teams_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1: return 0
    elif val == 2: return 1
    elif val == 3: return 2
    elif val == 4: return 3
    else: return np.nan

df["q78_no_sports_team_12m_bin"] = df["q78"].apply(sports_teams_bin).astype("float32")

def sm_use_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 1:         return 0
    elif val in [2,3,4]: return 1
    elif val in [5,6]:   return 2
    elif val in [7,8]:   return 3
    else: return np.nan

df["q80_freq_soc_media_bin"] = df["q80"].apply(sm_use_bin).astype("float32")

def sleep_hours_bin(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [1,2]:   return 0
    elif val in [3,4,5]: return 1
    elif val in [6,7]: return 2
    else: return np.nan

df["q85_hours_slept_bin"] = df["q85"].apply(sleep_hours_bin).astype("float32")

print("Binning complete.")

Binning complete.


### 4.4 Categorical Renamed Variables List and df_encoded_renamed

In [52]:
categorical_renamed_variables = [
    # Demographics
    'age_bin', 'grade', 'grade_label', 'Sex', 'sex_label', 'raceeth', 'raceeth_label',
    'sex_iden_gender_minority', 'sex_iden_gender_minority_label',

    # Violence exposure
    'q15_times_threatw_weapon_12m_bin', 'q16_times_in_phys_fight_12m_bin',
    'q17_times_fight_in_school_12m_bin', 'q18_seen_phy_viol_bin',
    'q19_forced_sex_bin', 'q88_adult_forced_sex_bin', 'q20_times_forced_sex_12m_bin',
    'q21_times_part_forced_sex_12m_bin', 'q22_times_part_hurt_12m_bin',
    'q89_times_insulted_byparent_l_bin', 'q90_times_hurt_byadult_l_bin',
    'q91_times_adults_phy_fought_30_bin',

    # Bullying
    'q23_times_treated_bad_atschool_race_bin', 'q23_treated_bad_atschool_race_ever',
    'q24_bullied_atschool_12m_bin', 'q25_cyber_bullied_12m_bin',
    'q105_unfairly_disciplined_12m_bin',

    # Tobacco use
    'q31_smoked_cig_bin', 'q32_age_1st_cig_bin', 'q33_days_smoked_cig_30_bin',
    'q36_days_vaped_30_bin', 'q38_days_tob_30_bin', 'q39_days_smoked_cigars_30_bin',
    'q34_no_of_cigs_pd_30_bin', 'q35_ever_vaped_bin', 'q40_tried_to_quit_12m_bin',

    # Alcohol use
    'q41_age_1st_drink_bin', 'q42_days_drank_alc_30_bin', 'q43_days_binge_alcdrink_30_bin',
    'q43_any_alcdrink_binge', 'q44_max_drinks_30_bin',

    # Drug use
    'q60_subs_before_lastsex_bin', 'q46_times_used_marijuana_l_bin',
    'q48_times_used_marijuana_30_bin', 'q49_times_pain_med_abuse_l_bin',
    'q92_times_pain_med_abuse_30_bin', 'q50_times_used_cocaine_l_bin',
    'q51_times_sniffed_aerosol_l_bin', 'q52_times_used_heroin_l_bin',
    'q53_times_used_methamph_l_bin', 'q54_times_used_ecstasy_l_bin',
    'q55_times_inject_drugs_l_bin', 'q93_times_used_hallucin_l_bin',
    'q47_age_1st_mari_bin',

    # Protective / social support factors
    'q9_times_rode_w_drunkdri_30_bin', 'q9_rode_wdrunkdriver_any',
    'q86_slept_where_30_bin', 'q86_stable_sleep',
    'q100_lived_w_guardian_w_drugalc_issues_l_bin',
    'q101_lived_w_severely_mentally_ill_guardian_l_bin',
    'q102_separated_from_guardian_l_bin',
    'q99_basic_needs_met_l_bin', 'q99_high_support', 'q104_adult_knew_whereabout_bin',
    'q87_grades_12m_bin', 'q87_high_grades', 'q103_feel_close_to_ppl_atschool_bin',
    'q83_last_dental_visit_bin', 'q83_recent_dental',

    # Diet, sleep, physical activity, social media
    'q68_juice_freq_bin', 'q69_fruit_freq_bin', 'q70_salad_freq_bin', 'q71_potato_freq_bin',
    'q72_carrot_freq_bin', 'q73_other_veg_freq_bin', 'q74_soda_freq_bin',
    'q75_breakfast_days_bin', 'q76_days_phys_active_7_bin', 'q77_days_pe_class_7_bin',
    'q78_no_sports_team_12m_bin', 'q80_freq_soc_media_bin', 'q85_hours_slept_bin',

    'mental_health_risk_category',
]

df_encoded_renamed = df[categorical_renamed_variables + ['weight']].copy()
df_encoded_renamed.replace(['Missing', 'Unknown'], np.nan, inplace=True)
df_encoded_renamed = df_encoded_renamed.loc[:, ~df_encoded_renamed.columns.duplicated(keep='last')]

df_encoded_renamed.head()

,age_bin,grade,grade_label,Sex,sex_label,raceeth,raceeth_label,sex_iden_gender_minority,sex_iden_gender_minority_label,q15_times_threatw_weapon_12m_bin,...,q73_other_veg_freq_bin,q74_soda_freq_bin,q75_breakfast_days_bin,q76_days_phys_active_7_bin,q77_days_pe_class_7_bin,q78_no_sports_team_12m_bin,q80_freq_soc_media_bin,q85_hours_slept_bin,mental_health_risk_category,weight
0,0.0,1.0,9th grade,1.0,Female,NaN,NaN,1,Sexual or Gender Minority,0.0,...,0.0,1.0,2.0,0.0,1.0,1.0,2.0,1.0,1,0.86
1,1.0,1.0,9th grade,2.0,Male,5.0,White,0,Majority,0.0,...,0.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,1,0.89
2,1.0,3.0,11th grade,2.0,Male,5.0,White,1,Sexual or Gender Minority,0.0,...,1.0,2.0,3.0,4.0,2.0,0.0,3.0,0.0,1,0.50
3,2.0,2.0,10th grade,1.0,Female,5.0,White,0,Majority,0.0,...,0.0,2.0,0.0,1.0,2.0,1.0,3.0,1.0,1,1.17
4,0.0,1.0,9th grade,2.0,Male,5.0,White,0,Majority,0.0,...,1.0,1.0,1.0,4.0,0.0,0.0,2.0,1.0,1,0.89


In [53]:
# Labeled version for inspection
df_labels_renamed = df_encoded_renamed.copy()
for var, mapping in value_labels.items():
    if var in df_labels_renamed.columns:
        df_labels_renamed[var] = df_labels_renamed[var].map(mapping)
df_labels_renamed.head()

,age_bin,grade,grade_label,Sex,sex_label,raceeth,raceeth_label,sex_iden_gender_minority,sex_iden_gender_minority_label,q15_times_threatw_weapon_12m_bin,...,q73_other_veg_freq_bin,q74_soda_freq_bin,q75_breakfast_days_bin,q76_days_phys_active_7_bin,q77_days_pe_class_7_bin,q78_no_sports_team_12m_bin,q80_freq_soc_media_bin,q85_hours_slept_bin,mental_health_risk_category,weight
0,Early adolescence (12-14),1.0,NaN,1.0,NaN,NaN,NaN,Sexual or Gender Minority,Sexual or Gender Minority,None,...,None,Low,Moderate,Inactive (0 days),1-2 days,1 team,Moderate,Adequate sleep,1,0.86
1,Middle adolescence (15-16),1.0,NaN,2.0,NaN,5.0,White,Majority,Majority,None,...,None,Low,Moderate,3-4 days,3+ days,1 team,Low,Adequate sleep,1,0.89
2,Middle adolescence (15-16),3.0,NaN,2.0,NaN,5.0,White,Sexual or Gender Minority,Sexual or Gender Minority,None,...,Low,Moderate,High,7 days,3+ days,No teams,High,Short sleep,1,0.50
3,Late adolescence (17-18+),2.0,NaN,1.0,NaN,5.0,White,Majority,Majority,None,...,None,Moderate,None,1-2 days,3+ days,1 team,High,Adequate sleep,1,1.17
4,Early adolescence (12-14),1.0,NaN,2.0,NaN,5.0,White,Majority,Majority,None,...,Low,Low,Low,7 days,None,No teams,Moderate,Adequate sleep,1,0.89


In [54]:
# Renamed variable information dataframe
variable_data_renamed = [
    {'Variable Name': 'age_bin',  'Meaning': 'Age',                      'Tag': 'Demographics'},
    {'Variable Name': 'sex',      'Meaning': 'Sex',                      'Tag': 'Demographics'},
    {'Variable Name': 'grade',    'Meaning': 'Grade',                    'Tag': 'Demographics'},
    {'Variable Name': 'raceeth',  'Meaning': 'race and ethinicity combined', 'Tag': 'Demographics'},

    {'Variable Name': 'q15_times_threatw_weapon_12m_bin',  'Meaning': 'Threatened with weapon at School',       'Tag': 'Violence Exposure'},
    {'Variable Name': 'q16_times_in_phys_fight_12m_bin',   'Meaning': 'In a physical fight',                    'Tag': 'Violence Exposure'},
    {'Variable Name': 'q17_times_fight_in_school_12m_bin', 'Meaning': 'In a physical fight at school',          'Tag': 'Violence Exposure'},
    {'Variable Name': 'q18_seen_phy_viol_bin',             'Meaning': 'Seen physical violence',                 'Tag': 'Violence Exposure'},
    {'Variable Name': 'q19_forced_sex_bin',                'Meaning': 'Physically Forced to have sex',          'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q20_times_forced_sex_12m_bin',      'Meaning': 'Times Forced to do sexual things',       'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q21_times_part_forced_sex_12m_bin', 'Meaning': 'Forced by partner to do sexual things',  'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q22_times_part_hurt_12m_bin',       'Meaning': 'Physically hurt by partner',             'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q88_adult_forced_sex_bin',          'Meaning': 'Sexual abuse by older adult',            'Tag': 'Sexual Violence, Violence Exposure'},
    {'Variable Name': 'q90_times_hurt_byadult_l_bin',      'Meaning': 'Parent/adult physical abuse (lifetime)', 'Tag': 'Parental Abuse, Violence Exposure'},

    {'Variable Name': 'q24_bullied_atschool_12m_bin',      'Meaning': 'Bullied at School',                      'Tag': 'Bullying'},
    {'Variable Name': 'q25_cyber_bullied_12m_bin',         'Meaning': 'Electronically(Cyber?) bullied',          'Tag': 'Bullying'},
    {'Variable Name': 'q105_unfairly_disciplined_12m_bin', 'Meaning': 'Unfairly disciplined at school (12 months)', 'Tag': 'Bullying'},

    {'Variable Name': 'q31_smoked_cig_bin',          'Meaning': 'Ever smoked cigarette',              'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q32_age_1st_cig_bin',         'Meaning': 'Age first smoked cigarette',         'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q33_days_smoked_cig_30_bin',  'Meaning': 'Days smoked cigarettes (30 days)',   'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q34_no_of_cigs_pd_30_bin',    'Meaning': 'Cigs/day on days smoked',           'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q35_ever_vaped_bin',          'Meaning': 'Ever used electronic vapor product', 'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q36_days_vaped_30_bin',       'Meaning': 'Days used vapor product (30 days)', 'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q38_days_tob_30_bin',         'Meaning': 'Days used chewing tobacco (30 days)','Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q39_days_smoked_cigars_30_bin','Meaning': 'Days smoked cigars (30 days)',      'Tag': 'Tobacco/Nicotine Use'},
    {'Variable Name': 'q40_tried_to_quit_12m_bin',   'Meaning': 'Tried to quit all tobacco (12 months)', 'Tag': 'Tobacco/Nicotine Use'},

    {'Variable Name': 'q41_age_1st_drink_bin',       'Meaning': 'Age first drank alcohol',           'Tag': 'Alcohol Use'},
    {'Variable Name': 'q42_days_drank_alc_30_bin',   'Meaning': 'Days drank alcohol (30 days)',      'Tag': 'Alcohol Use'},
    {'Variable Name': 'q43_days_binge_alcdrink_30_bin','Meaning': 'Days had 4+/5+ drinks in row (30 days)', 'Tag': 'Alcohol Use'},
    {'Variable Name': 'q44_max_drinks_30_bin',        'Meaning': 'Max drinks in a row (30 days)',    'Tag': 'Alcohol Use'},
    {'Variable Name': 'q60_subs_before_lastsex_bin',  'Meaning': 'Used alcohol/drugs before last sex','Tag': 'Alcohol Use'},

    {'Variable Name': 'q46_times_used_marijuana_l_bin',  'Meaning': 'Times used marijuana (lifetime)',           'Tag': 'Other Drugs'},
    {'Variable Name': 'q47_age_1st_mari_bin',            'Meaning': 'Age first tried marijuana',                 'Tag': 'Other Drugs'},
    {'Variable Name': 'q48_times_used_marijuana_30_bin', 'Meaning': 'Times used marijuana (30 days)',            'Tag': 'Other Drugs'},
    {'Variable Name': 'q49_times_pain_med_abuse_l_bin',  'Meaning': 'Times misused pain medicine during lifetime','Tag': 'Other Drugs'},
    {'Variable Name': 'q92_times_pain_med_abuse_30_bin', 'Meaning': 'Times misused pain medicine during past 30 days', 'Tag': 'Other Drugs'},
    {'Variable Name': 'q50_times_used_cocaine_l_bin',    'Meaning': 'Times used cocaine',                        'Tag': 'Other Drugs'},
    {'Variable Name': 'q51_times_sniffed_aerosol_l_bin', 'Meaning': 'Times used inhalants',                      'Tag': 'Other Drugs'},
    {'Variable Name': 'q52_times_used_heroin_l_bin',     'Meaning': 'Times used heroin',                         'Tag': 'Other Drugs'},
    {'Variable Name': 'q53_times_used_methamph_l_bin',   'Meaning': 'Times used methamphetamines',               'Tag': 'Other Drugs'},
    {'Variable Name': 'q54_times_used_ecstasy_l_bin',    'Meaning': 'Times used ecstasy',                        'Tag': 'Other Drugs'},
    {'Variable Name': 'q55_times_inject_drugs_l_bin',    'Meaning': 'Times injected illegal drugs',              'Tag': 'Other Drugs'},
    {'Variable Name': 'q93_times_used_hallucin_l_bin',   'Meaning': 'Times used hallucinogens (lifetime)',       'Tag': 'Other Drugs'},

    {'Variable Name': 'q9_times_rode_w_drunkdri_30_bin', 'Meaning': 'Times rode with drunk driver (30 days)',    'Tag': 'ACEs'},
    {'Variable Name': 'q86_slept_where_30_bin',          'Meaning': 'Slept usually where (30 days)',             'Tag': 'Sleep, ACEs'},
    {'Variable Name': 'q89_times_insulted_byparent_l_bin','Meaning': 'Parent/adult insulted/put down (lifetime)', 'Tag': 'Parental, ACEs'},
    {'Variable Name': 'q91_times_adults_phy_fought_30_bin','Meaning': 'Parents/adults fought each other (lifetime)','Tag': 'Parental, ACEs'},
    {'Variable Name': 'q100_lived_w_guardian_w_drugalc_issues_l_bin',      'Meaning': 'Lived with addict parent/guardian',     'Tag': 'Parental Addiction, ACEs'},
    {'Variable Name': 'q101_lived_w_severely_mentally_ill_guardian_l_bin', 'Meaning': 'Lived with mentally ill parent/guardian','Tag': 'Parental Mental Health, ACEs'},
    {'Variable Name': 'q102_separated_from_guardian_l_bin','Meaning': 'Separated from parent because of incarceration',       'Tag': 'Parental Separation, ACEs'},

    {'Variable Name': 'q99_basic_needs_met_l_bin',        'Meaning': 'Adult met basic needs (lifetime)',   'Tag': 'Parental, Support'},
    {'Variable Name': 'q104_adult_knew_whereabout_bin',   'Meaning': 'An Adult knows whereabouts',        'Tag': 'Supervision, Support'},
    {'Variable Name': 'q87_grades_12m_bin',               'Meaning': 'Grades in school (12 months)',      'Tag': 'Academic, Support'},
    {'Variable Name': 'q103_feel_close_to_ppl_atschool_bin','Meaning': 'Feel close to people at school',  'Tag': 'School, Support'},
    {'Variable Name': 'q83_last_dental_visit_bin',        'Meaning': 'Last dental visit',                 'Tag': 'Medical, Support'},

    {'Variable Name': 'q85_hours_slept_bin',              'Meaning': 'Sleep hours (school night)',        'Tag': 'Sleep'},
    {'Variable Name': 'q76_days_phys_active_7_bin',       'Meaning': 'Days active 60+ min (7 days)',      'Tag': 'Physical Activity'},
    {'Variable Name': 'q77_days_pe_class_7_bin',          'Meaning': 'PE class days (week)',              'Tag': 'Physical Activity'},
    {'Variable Name': 'q78_no_sports_team_12m_bin',       'Meaning': 'Sports teams played (12 months)',   'Tag': 'Physical Activity'},

    {'Variable Name': 'q68_juice_freq_bin',               'Meaning': 'Drank 100% fruit juice (7 days)',  'Tag': 'Diet'},
    {'Variable Name': 'q69_fruit_freq_bin',               'Meaning': 'Ate fruit (7 days)',               'Tag': 'Diet'},
    {'Variable Name': 'q70_salad_freq_bin',               'Meaning': 'Ate green salad (7 days)',         'Tag': 'Diet'},
    {'Variable Name': 'q71_potato_freq_bin',              'Meaning': 'Ate potatoes (7 days)',            'Tag': 'Diet'},
    {'Variable Name': 'q72_carrot_freq_bin',              'Meaning': 'Ate carrots (7 days)',             'Tag': 'Diet'},
    {'Variable Name': 'q73_other_veg_freq_bin',           'Meaning': 'Ate vegetables (7 days)',          'Tag': 'Diet'},
    {'Variable Name': 'q74_soda_freq_bin',                'Meaning': 'Drank soda/pop (7 days)',          'Tag': 'Diet'},
    {'Variable Name': 'q75_breakfast_days_bin',           'Meaning': 'Days ate breakfast (7 days)',      'Tag': 'Diet'},

    {'Variable Name': 'q80_freq_soc_media_bin',           'Meaning': 'Social media use frequency',      'Tag': 'Social'},
    {'Variable Name': 'q23_times_treated_bad_atschool_race_bin','Meaning': 'Treated badly because of race/ethnicity','Tag': 'Race, Bullying'},
]

## 5. Missing Data Analysis

In [55]:
def weighted_missing_rate(df, group_mask, col, weight_col):
    """Weighted proportion of missing values in `col` within rows where group_mask is True."""
    sub = df.loc[group_mask & df[weight_col].notna(), [col, weight_col]].copy()
    if sub.empty:
        return np.nan
    w = sub[weight_col].to_numpy()
    m = sub[col].isna().to_numpy().astype(float)
    denom = w.sum()
    if denom == 0:
        return np.nan
    return float((w * m).sum() / denom)


def missing_by_subgroup_weighted_generic(df,
                                          subgroup_col,
                                          weight_col,
                                          level_ref=None,
                                          level_comp=None):
    """
    Compute weighted missingness by subgroup (2-category) for every column.

    Parameters
    ----------
    df           : DataFrame
    subgroup_col : str   Column defining the subgroup
    weight_col   : str   Survey weight column
    level_ref    : optional  Reference level; defaults to most frequent non-missing level
    level_comp   : optional  Comparison level; defaults to second most frequent level

    Returns
    -------
    DataFrame with weighted missingness stats for each variable.
    """
    df_sub = df[df[subgroup_col].notna() & df[weight_col].notna()].copy()

    levels = df_sub[subgroup_col].value_counts().index.tolist()
    if len(levels) < 2:
        raise ValueError(f"{subgroup_col} has fewer than 2 non-missing levels")

    if level_ref is None:  level_ref  = levels[0]
    if level_comp is None: level_comp = levels[1]

    print(f"Comparing missingness by {subgroup_col}: '{level_comp}' vs '{level_ref}'")

    results = []
    for col in df_sub.columns:
        if col in [subgroup_col, weight_col]:
            continue

        overall_mask = df_sub[col].isna().to_numpy().astype(float)
        w_all = df_sub[weight_col].to_numpy()
        denom_all = w_all.sum()
        total_miss = float((overall_mask * w_all).sum() / denom_all) if denom_all > 0 else np.nan

        ref_mask  = (df_sub[subgroup_col] == level_ref)
        comp_mask = (df_sub[subgroup_col] == level_comp)
        miss_ref  = weighted_missing_rate(df_sub, ref_mask,  col, weight_col)
        miss_comp = weighted_missing_rate(df_sub, comp_mask, col, weight_col)

        if pd.isna(miss_ref) or pd.isna(miss_comp):
            diff_abs = np.nan
            diff_rel = np.nan
        else:
            diff_abs = miss_comp - miss_ref
            diff_rel = (diff_abs / miss_ref * 100.0) if miss_ref > 0 else np.nan

        results.append({
            "variable": col,
            "group_var": subgroup_col,
            "ref_level": level_ref,
            "comp_level": level_comp,
            "total_weight": denom_all,
            "missing_overall_pct": total_miss * 100.0 if not pd.isna(total_miss) else np.nan,
            f"missing_{level_ref}_pct": miss_ref * 100.0 if not pd.isna(miss_ref) else np.nan,
            f"missing_{level_comp}_pct": miss_comp * 100.0 if not pd.isna(miss_comp) else np.nan,
            "abs_diff_pct": diff_abs * 100.0 if not pd.isna(diff_abs) else np.nan,
            "rel_diff_pct": diff_rel
        })

    res_df = pd.DataFrame(results)
    res_df = res_df.sort_values("abs_diff_pct", ascending=False)
    return res_df

In [56]:
# Missing data comparisons across key demographic subgroups
miss_lgbt = missing_by_subgroup_weighted_generic(
    df_encoded_renamed,
    subgroup_col="sex_iden_gender_minority_label",
    weight_col="weight",
    level_ref="Majority",
    level_comp="Sexual or Gender Minority"
)

miss_sex = missing_by_subgroup_weighted_generic(
    df_encoded_renamed,
    subgroup_col="sex_label",
    weight_col="weight",
    level_ref="Male",
    level_comp="Female"
)

miss_race = missing_by_subgroup_weighted_generic(
    df_encoded_renamed,
    subgroup_col="raceeth_label",
    weight_col="weight",
    level_ref="White",
    level_comp="Black or African American"
)

miss_grade = missing_by_subgroup_weighted_generic(
    df_encoded_renamed,
    subgroup_col="grade_label",
    weight_col="weight",
    level_ref="9th grade",
    level_comp="12th grade"
)

Comparing missingness by sex_iden_gender_minority_label: 'Sexual or Gender Minority' vs 'Majority'
Comparing missingness by sex_label: 'Female' vs 'Male'
Comparing missingness by raceeth_label: 'Black or African American' vs 'White'
Comparing missingness by grade_label: '12th grade' vs '9th grade'


In [57]:
# Top 5 missing variables per subgroup
for label, miss_df in [("LGBT", miss_lgbt), ("Sex", miss_sex), ("Race", miss_race), ("Grade", miss_grade)]:
    print(f"=== {label} ===")
    print(miss_df.head(5).to_string(index=False))
    print()

=== LGBT ===
                         variable                      group_var ref_level                comp_level  total_weight  missing_overall_pct  missing_Majority_pct  missing_Sexual or Gender Minority_pct  abs_diff_pct  rel_diff_pct
     q20_times_forced_sex_12m_bin sex_iden_gender_minority_label  Majority Sexual or Gender Minority    18642.7759            17.330752             15.529434                              22.753345      7.223911     46.517546
q21_times_part_forced_sex_12m_bin sex_iden_gender_minority_label  Majority Sexual or Gender Minority    18642.7759            15.754342             14.155997                              20.565917      6.409919     45.280592
  q92_times_pain_med_abuse_30_bin sex_iden_gender_minority_label  Majority Sexual or Gender Minority    18642.7759            27.758083             26.372897                              31.927977      5.555081     21.063597
               q31_smoked_cig_bin sex_iden_gender_minority_label  Majority Sexual or Ge

In [58]:
miss_sex.head(15)

,variable,group_var,ref_level,comp_level,total_weight,missing_overall_pct,missing_Male_pct,missing_Female_pct,abs_diff_pct,rel_diff_pct
15,q21_times_part_forced_sex_12m_bin,sex_label,Male,Female,19849.2932,18.336353,16.752874,20.044825,3.291951,19.650069
14,q20_times_forced_sex_12m_bin,sex_label,Male,Female,19849.2932,19.917361,18.450630,21.499867,3.049237,16.526465
43,q92_times_pain_med_abuse_30_bin,sex_label,Male,Female,19849.2932,30.715318,29.590300,31.929139,2.338840,7.904075
33,q40_tried_to_quit_12m_bin,sex_label,Male,Female,19849.2932,19.616496,18.630295,20.680543,2.050248,11.004915
19,q91_times_adults_phy_fought_30_bin,sex_label,Male,Female,19849.2932,19.526610,18.548931,20.581462,2.032531,10.957671
12,q19_forced_sex_bin,sex_label,Male,Female,19849.2932,13.363321,12.419592,14.381543,1.961951,15.797227
11,q18_seen_phy_viol_bin,sex_label,Male,Female,19849.2932,21.794470,20.853790,22.809403,1.955614,9.377737
17,q89_times_insulted_byparent_l_bin,sex_label,Male,Female,19849.2932,17.985995,17.266902,18.761849,1.494947,8.657876
13,q88_adult_forced_sex_bin,sex_label,Male,Female,19849.2932,19.076187,18.385600,19.821286,1.435686,7.808752
24,q105_unfairly_disciplined_12m_bin,sex_label,Male,Female,19849.2932,35.737111,35.078633,36.447567,1.368934,3.902473


In [59]:
# Variables with >= 5 percentage point differential missing by subgroup
serious_lgbt  = miss_lgbt[miss_lgbt["abs_diff_pct"].abs() >= 5]
serious_sex   = miss_sex[miss_sex["abs_diff_pct"].abs() >= 5]
serious_race  = miss_race[miss_race["abs_diff_pct"].abs() >= 5]
serious_grade = miss_grade[miss_grade["abs_diff_pct"].abs() >= 5]

def summarize_missing_gaps(miss_df, threshold=5):
    serious = miss_df[miss_df["abs_diff_pct"].abs() >= threshold]
    return {
        "group_var":        serious["group_var"].iloc[0] if not serious.empty else miss_df["group_var"].iloc[0],
        "n_vars_large_gap": serious.shape[0],
        "mean_abs_gap":     serious["abs_diff_pct"].abs().mean(),
        "max_abs_gap":      serious["abs_diff_pct"].abs().max(),
    }

summary_rows = [
    summarize_missing_gaps(miss_lgbt),
    summarize_missing_gaps(miss_sex),
    summarize_missing_gaps(miss_race),
    summarize_missing_gaps(miss_grade),
]
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

                        group_var  n_vars_large_gap  mean_abs_gap  max_abs_gap
0  sex_iden_gender_minority_label                 3      6.396304     7.223911
1                       sex_label                 0           NaN          NaN
2                   raceeth_label                37     10.255829    18.562591
3                     grade_label                 2      5.499307     5.602688


### 5.1 Missingness Heatmap

> Heatmap of weighted missingness rates for all analysis variables across four subgroup comparisons (LGBT, Sex, Race, Grade). Darker colors indicate higher missingness.

In [ ]:
# ── Missingness Heatmap: Overall + By Subgroup ──────────────────────────────
#
# Builds a single heatmap showing weighted missing % for every analysis variable
# across four comparisons: LGBT status, Sex, Race, and Grade.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Combine the four missingness DataFrames into one wide table
def build_missing_heatmap_data(miss_dfs, labels):
    """Merge multiple miss_* DataFrames into one matrix for heatmap."""
    base = miss_dfs[0][['variable', 'missing_overall_pct']].copy()
    base = base.rename(columns={'missing_overall_pct': 'Overall'})
    
    for mdf, label in zip(miss_dfs, labels):
        ref_col = [c for c in mdf.columns if c.startswith('missing_') and c.endswith('_pct') and 'overall' not in c][0]
        comp_col = [c for c in mdf.columns if c.startswith('missing_') and c.endswith('_pct') and 'overall' not in c][1]
        sub = mdf[['variable', ref_col, comp_col]].copy()
        sub = sub.rename(columns={ref_col: f'{label} (Ref)', comp_col: f'{label} (Comp)'})
        base = base.merge(sub, on='variable', how='outer')
    
    base = base.set_index('variable')
    # Sort by overall missingness descending
    base = base.sort_values('Overall', ascending=True)
    return base

heatmap_data = build_missing_heatmap_data(
    [miss_lgbt, miss_sex, miss_race, miss_grade],
    ['LGBT', 'Sex', 'Race', 'Grade']
)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns.tolist(),
    y=heatmap_data.index.tolist(),
    colorscale='YlOrRd',
    colorbar=dict(title='Missing %'),
    text=np.round(heatmap_data.values, 1),
    texttemplate='%{text:.1f}',
    textfont=dict(size=8),
    hovertemplate='Variable: %{y}<br>Subgroup: %{x}<br>Missing: %{z:.1f}%<extra></extra>',
))

fig.update_layout(
    title='Weighted Missingness (%) Across Variables and Demographic Subgroups',
    xaxis_title='Subgroup Comparison',
    yaxis_title='Variable',
    height=max(600, len(heatmap_data) * 18),
    width=900,
    margin=dict(l=280),
    yaxis=dict(tickfont=dict(size=9)),
)
fig.show()


### 5.2 Differential Missingness by Subgroup

> Variables with the largest absolute difference in missingness between subgroup levels. Only variables with >= 5 percentage point gap are shown.

In [ ]:
# ── Differential Missingness Bar Charts ──────────────────────────────────────
#
# One subplot per subgroup showing top variables with >= 5 pp differential.

from plotly.subplots import make_subplots

subgroup_data = [
    ('LGBT Status', miss_lgbt, 'Majority', 'SGM'),
    ('Sex', miss_sex, 'Male', 'Female'),
    ('Race (White vs Black)', miss_race, 'White', 'Black'),
    ('Grade (9th vs 12th)', miss_grade, '9th', '12th'),
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[s[0] for s in subgroup_data],
    horizontal_spacing=0.15,
    vertical_spacing=0.25,
)

for idx, (title, mdf, ref_label, comp_label) in enumerate(subgroup_data):
    row, col = (idx // 2) + 1, (idx % 2) + 1
    
    # Get variables with >= 5 pp absolute difference, top 10
    serious = mdf[mdf['abs_diff_pct'].abs() >= 5].copy()
    serious = serious.nlargest(10, 'abs_diff_pct')
    
    if serious.empty:
        continue
    
    # Get the ref and comp column names
    ref_col = [c for c in mdf.columns if c.startswith('missing_') and c.endswith('_pct') and 'overall' not in c][0]
    comp_col = [c for c in mdf.columns if c.startswith('missing_') and c.endswith('_pct') and 'overall' not in c][1]
    
    # Shorten variable names for readability
    var_short = serious['variable'].str.replace('_bin$', '', regex=True).str[:30]
    
    fig.add_trace(
        go.Bar(name=ref_label, x=serious[ref_col].values, y=var_short,
               orientation='h', marker_color='#1f77b4',
               showlegend=(idx == 0)),
        row=row, col=col
    )
    fig.add_trace(
        go.Bar(name=comp_label, x=serious[comp_col].values, y=var_short,
               orientation='h', marker_color='#d62728',
               showlegend=(idx == 0)),
        row=row, col=col
    )

fig.update_layout(
    title='Top Variables with Differential Missingness by Subgroup (≥5 pp gap)',
    barmode='group',
    height=700,
    width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
)
fig.update_xaxes(title_text='Weighted Missing %')
fig.show()


### 5.3 Overall Missingness Distribution

> Overall weighted missingness rate for every analysis variable, colored by domain.

In [ ]:
# ── Overall Missingness by Variable (colored by domain) ──────────────────────

# Domain assignment based on variable name patterns
def assign_domain(var):
    if any(k in var for k in ['age_bin', 'grade', 'Sex', 'sex_label', 'raceeth', 'sex_iden']):
        return 'Demographics'
    if any(k in var for k in ['q15_', 'q16_', 'q17_', 'q18_', 'q19_', 'q20_', 'q21_', 'q22_', 'q88_', 'q90_']):
        return 'Violence / Sexual Violence'
    if any(k in var for k in ['q23_', 'q24_', 'q25_', 'q105_']):
        return 'Bullying'
    if any(k in var for k in ['q31_', 'q32_', 'q33_', 'q34_', 'q35_', 'q36_', 'q38_', 'q39_', 'q40_']):
        return 'Tobacco / Nicotine'
    if any(k in var for k in ['q41_', 'q42_', 'q43_', 'q44_', 'q60_']):
        return 'Alcohol'
    if any(k in var for k in ['q46_', 'q47_', 'q48_', 'q49_', 'q50_', 'q51_', 'q52_', 'q53_', 'q54_', 'q55_', 'q92_', 'q93_']):
        return 'Drugs'
    if any(k in var for k in ['q89_', 'q91_', 'q100_', 'q101_', 'q102_', 'q9_', 'q86_']):
        return 'ACEs / Parental'
    if any(k in var for k in ['q99_', 'q104_', 'q87_', 'q103_', 'q83_']):
        return 'Protective Factors'
    if any(k in var for k in ['q68_', 'q69_', 'q70_', 'q71_', 'q72_', 'q73_', 'q74_', 'q75_', 'q76_', 'q77_', 'q78_', 'q80_', 'q85_']):
        return 'Lifestyle'
    return 'Other'

miss_overall = miss_lgbt[['variable', 'missing_overall_pct']].copy()
miss_overall['domain'] = miss_overall['variable'].apply(assign_domain)
miss_overall = miss_overall.sort_values('missing_overall_pct', ascending=True)

# Color palette per domain
domain_colors = {
    'Demographics': '#636EFA', 'Violence / Sexual Violence': '#EF553B',
    'Bullying': '#00CC96', 'Tobacco / Nicotine': '#AB63FA',
    'Alcohol': '#FFA15A', 'Drugs': '#19D3F3',
    'ACEs / Parental': '#FF6692', 'Protective Factors': '#B6E880',
    'Lifestyle': '#FF97FF', 'Other': '#FECB52',
}

fig = go.Figure()
for domain in miss_overall['domain'].unique():
    sub = miss_overall[miss_overall['domain'] == domain]
    fig.add_trace(go.Bar(
        y=sub['variable'], x=sub['missing_overall_pct'],
        name=domain, orientation='h',
        marker_color=domain_colors.get(domain, '#999'),
        hovertemplate='%{y}<br>Missing: %{x:.1f}%<extra></extra>',
    ))

fig.add_vline(x=25, line_dash='dash', line_color='red', annotation_text='25% threshold')

fig.update_layout(
    title='Overall Weighted Missingness by Variable (colored by domain)',
    xaxis_title='Weighted Missing %',
    yaxis_title='Variable',
    barmode='stack',
    height=max(600, len(miss_overall) * 18),
    width=900,
    margin=dict(l=280),
    yaxis=dict(tickfont=dict(size=9)),
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5),
)
fig.show()


### 5.4 Feature Redundancy Check

In [60]:
def check_variable_redundancy(df, variables):
    """Check if variables are redundant (highly overlapping) using Cramer's V."""
    print("Variable redundancy check")
    for i, v1 in enumerate(variables):
        for v2 in variables[i+1:]:
            ct = pd.crosstab(df[v1], df[v2], dropna=False)
            try:
                chi2, p_val, dof, expected = chi2_contingency(ct)
                n = ct.sum().sum()
                cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
                if cramers_v > 0.7:   decision = "Highly redundant"
                elif cramers_v > 0.5: decision = "Moderately redundant"
                elif cramers_v > 0.3: decision = "Similar"
                else:                 decision = "Distinct"
                print(f"\n{v1[:40]:40s} x {v2[:40]:40s}")
                print(f"  Cramer's V = {cramers_v:.3f}, p = {p_val:.4f} -> {decision}")
            except Exception as e:
                print(f"\n{v1} x {v2}: Error - {e}")
    print("\n")

check_variable_redundancy(
    df_encoded_renamed,
    [
        'q16_times_in_phys_fight_12m_bin',   # 42% missing
        'q17_times_fight_in_school_12m_bin',
        'q92_times_pain_med_abuse_30_bin',
        'q49_times_pain_med_abuse_l_bin',
        'q93_times_used_hallucin_l_bin',     # 44.4% missing
        'q42_days_drank_alc_30_bin',
        'q43_days_binge_alcdrink_30_bin',
        'q46_times_used_marijuana_l_bin',
        'q48_times_used_marijuana_30_bin',
    ]
)

Variable redundancy check

q16_times_in_phys_fight_12m_bin          x q17_times_fight_in_school_12m_bin       
  Cramer's V = 0.358, p = 0.0000 -> Similar

q16_times_in_phys_fight_12m_bin          x q92_times_pain_med_abuse_30_bin         
  Cramer's V = 0.410, p = 0.0000 -> Similar

q16_times_in_phys_fight_12m_bin          x q49_times_pain_med_abuse_l_bin          
  Cramer's V = 0.084, p = 0.0000 -> Distinct

q16_times_in_phys_fight_12m_bin          x q93_times_used_hallucin_l_bin           
  Cramer's V = 0.495, p = 0.0000 -> Similar

q16_times_in_phys_fight_12m_bin          x q42_days_drank_alc_30_bin               
  Cramer's V = 0.128, p = 0.0000 -> Distinct

q16_times_in_phys_fight_12m_bin          x q43_days_binge_alcdrink_30_bin          
  Cramer's V = 0.268, p = 0.0000 -> Distinct

q16_times_in_phys_fight_12m_bin          x q46_times_used_marijuana_l_bin          
  Cramer's V = 0.251, p = 0.0000 -> Distinct

q16_times_in_phys_fight_12m_bin          x q48_times_used_marijuan

In [61]:
# Features with extremely high missingness (>25%)
extremely_missing_features = df_encoded_renamed.columns[df_encoded_renamed.isnull().mean() > 0.25]
print(extremely_missing_features)
df_encoded_renamed.info()

Index(['q16_times_in_phys_fight_12m_bin', 'q18_seen_phy_viol_bin',
       'q88_adult_forced_sex_bin', 'q89_times_insulted_byparent_l_bin',
       'q90_times_hurt_byadult_l_bin', 'q91_times_adults_phy_fought_30_bin',
       'q105_unfairly_disciplined_12m_bin', 'q34_no_of_cigs_pd_30_bin',
       'q44_max_drinks_30_bin', 'q92_times_pain_med_abuse_30_bin',
       'q51_times_sniffed_aerosol_l_bin', 'q93_times_used_hallucin_l_bin',
       'q86_slept_where_30_bin', 'q86_stable_sleep',
       'q100_lived_w_guardian_w_drugalc_issues_l_bin',
       'q101_lived_w_severely_mentally_ill_guardian_l_bin',
       'q102_separated_from_guardian_l_bin', 'q104_adult_knew_whereabout_bin',
       'q103_feel_close_to_ppl_atschool_bin', 'q75_breakfast_days_bin',
       'q77_days_pe_class_7_bin', 'q78_no_sports_team_12m_bin'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20103 entries, 0 to 20102
Data columns (total 83 columns):
 #   Column                                            

### 5.5 Variables to Drop for Modeling

In [ ]:
vars_to_drop_for_modeling = [
    # Heavy missingness and weakest theoretical leverage relative to close alternatives
    "q105_unfairly_disciplined_12m_bin",    # high missing; weaker than bullying + race items
    "q93_times_used_hallucin_l_bin",        # rare hallucinogen use; lifetime with ~32% missing
    "q92_times_pain_med_abuse_30_bin",      # keeping lifetime misuse (q49) as core opioid risk
    "q34_no_of_cigs_pd_30_bin",             # already have smoking days + ever smoked
    "q51_times_sniffed_aerosol_l_bin",      # rare inhalant use; high missing
]
df_encoded_renamed = df_encoded_renamed.drop(columns=vars_to_drop_for_modeling)
print(f"Dataset shape after dropping: {df_encoded_renamed.shape}")

## 6. Project 4: Baseline Modeling (LightGBM)

In [ ]:
# Build df_lgbm with categorical dtype for LightGBM native handling
df_lgbm = df[categorical_renamed_variables].copy()
df_lgbm = df_lgbm.loc[:, ~df_lgbm.columns.duplicated()]

for col in df_lgbm.columns:
    if col not in ['mental_health_risk_category', 'weight']:
        df_lgbm[col] = df_lgbm[col].astype('category')

X = df_lgbm.drop(columns=['mental_health_risk_category', 'weight']).copy()
y = df_lgbm['mental_health_risk_category']

weights = df_lgbm['weight'].values
weights = np.asarray(weights, dtype=np.float32).ravel()

categorical_feature_names = [col for col in X.columns if X[col].dtype.name == 'category']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, stratify=y, random_state=42
)
w_train = w_train.tolist()
w_test  = w_test.tolist()

lgbm = LGBMClassifier(
    n_estimators=300, max_depth=20, learning_rate=0.05, random_state=42
)
lgbm.fit(X_train, y_train, sample_weight=w_train, categorical_feature=categorical_feature_names)

y_pred = lgbm.predict(X_test)
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

feature_importance_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': lgbm.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 20 Original Features:")
print(feature_importance_df.head(20))

In [ ]:
# Combined survey + class weights model
classes       = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
sample_class_weights = np.array([class_weights[int(y)] for y in y_train])

# Combine survey weights with class weights
w_train_combined = w_train * sample_class_weights
w_train_combined = pd.Series(w_train_combined, dtype=np.float32).reset_index(drop=True)

lgbm = LGBMClassifier(
    n_estimators=700, max_depth=12, learning_rate=0.03, num_leaves=31,
    min_child_samples=25, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1
)
lgbm.fit(X_train, y_train, sample_weight=w_train_combined,
         categorical_feature=categorical_feature_names)

y_pred = lgbm.predict(X_test)
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

feature_importance_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': lgbm.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 20 Original Features:")
print(feature_importance_df.head(20))

In [ ]:
# Grid search for optimal hyperparameters
param_grid = {
    'n_estimators':      [700],
    'max_depth':         [8, 10, 12],
    'learning_rate':     [0.01, 0.05],
    'num_leaves':        [31, 50],
    'min_child_samples': [20, 40],
    'subsample':         [0.8],
    'colsample_bytree':  [0.8],
    'reg_alpha':         [0, 0.1],
    'reg_lambda':        [0, 0.1],
}

lgbm_gs = LGBMClassifier(random_state=42, verbose=-1)
grid_search = GridSearchCV(
    lgbm_gs, param_grid, cv=3, scoring='balanced_accuracy', n_jobs=-1, verbose=2
)
grid_search.fit(X_train, y_train, sample_weight=w_train_combined)

print("Best parameters:", grid_search.best_params_)
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
print("\nBalanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Optimized model with top 30 features and adjusted class weights
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights[1] *= 1.5  # Moderate risk: extra emphasis
class_weights[2] *= 2.0  # High risk: highest priority

sample_class_weights = np.array([class_weights[int(y)] for y in y_train])
w_train_combined = w_train * sample_class_weights
w_train_combined = pd.Series(w_train_combined, dtype=np.float32).reset_index(drop=True)

print(f"Class weights applied: {class_weights}")
print(f"Class distribution - Train: {np.bincount(y_train.astype(int))}")

lgbm_optimized = LGBMClassifier(
    n_estimators=1500, max_depth=10, learning_rate=0.01, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=0.1, min_split_gain=0.02, min_data_per_group=30,
    subsample_freq=1, objective='multiclass', num_class=3, boosting_type='gbdt',
    random_state=42, verbose=-1, n_jobs=-1
)

lgbm_optimized.fit(
    X_train, y_train,
    sample_weight=w_train_combined,
    categorical_feature=categorical_feature_names,
    eval_set=[(X_test, y_test)],
    eval_metric='multi_logloss',
    callbacks=[lgb.early_stopping(150, verbose=False)]
)

print(f"\nBest iteration: {lgbm_optimized.best_iteration_}")
print(f"Validation multi-logloss: {lgbm_optimized.best_score_['valid_0']['multi_logloss']:.4f}")

y_pred_baseline = lgbm_optimized.predict(X_test)
bal_acc_baseline = balanced_accuracy_score(y_test, y_pred_baseline)

print("BASELINE PERFORMANCE (Default Thresholds)")
print(f"Balanced accuracy: {bal_acc_baseline:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline))

print("OPTIMIZING DECISION THRESHOLDS FOR MINORITY CLASSES")
y_proba     = lgbm_optimized.predict_proba(X_test)
best_score  = 0
best_thresholds = None
best_metrics    = None

print("\nSearching for optimal thresholds...")
for t0 in np.arange(0.4, 0.7, 0.05):
    for t1 in np.arange(0.25, 0.45, 0.05):
        for t2 in np.arange(0.25, 0.45, 0.05):
            y_proba_adj = y_proba / np.array([t0, t1, t2])
            y_pred_adj  = np.argmax(y_proba_adj, axis=1)
            bal_acc = balanced_accuracy_score(y_test, y_pred_adj)
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_test, y_pred_adj, average=None, zero_division=0
            )
            # Prioritize minority class recall and balanced accuracy
            score = bal_acc * 1.0 + rec[2] * 0.3 + prec[2] * 0.4 + f1[2] * 0.3
            if score > best_score:
                best_score = score
                best_thresholds = (t0, t1, t2)
                best_metrics = {'bal_acc': bal_acc, 'precision': prec, 'recall': rec, 'f1': f1}

print(f"\nOptimal thresholds found: {best_thresholds}")
print(f"  Class 0 (Low risk):      {best_thresholds[0]:.2f}")
print(f"  Class 1 (Moderate risk): {best_thresholds[1]:.2f}")
print(f"  Class 2 (High risk):     {best_thresholds[2]:.2f}")

y_proba_optimized = y_proba / np.array(best_thresholds)
y_pred_optimized  = np.argmax(y_proba_optimized, axis=1)

print("\nFINAL PERFORMANCE (Optimized Thresholds)")
print(f"Balanced accuracy: {best_metrics['bal_acc']:.4f}")
print(f"Improvement: +{(best_metrics['bal_acc'] - bal_acc_baseline):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_optimized))
print("\nPer-Class Metrics:")
for i, class_name in enumerate(['Low Risk (0)', 'Moderate Risk (1)', 'High Risk (2)']):
    print(f"\n{class_name}:")
    print(f"  Precision: {best_metrics['precision'][i]:.3f}")
    print(f"  Recall:    {best_metrics['recall'][i]:.3f}")
    print(f"  F1-Score:  {best_metrics['f1'][i]:.3f}")

feature_importance_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': lgbm_optimized.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTOP 20 MOST IMPORTANT FEATURES")
print(feature_importance_df.head(20).to_string(index=False))
top_features_baseline = feature_importance_df.head(20)['feature'].tolist()
print(f"\nTop features list for stratified analysis:")
print(top_features_baseline)

cm = confusion_matrix(y_test, y_pred_optimized)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print("\nCONFUSION MATRIX (Normalized by True Class)")
print("          Predicted")
print("          Low   Mod   High")
print("True")
for i, label in enumerate(['Low  ', 'Mod  ', 'High ']):
    print(f"{label}  {cm_normalized[i,0]:.2f}  {cm_normalized[i,1]:.2f}  {cm_normalized[i,2]:.2f}  (n={cm[i].sum()})")

print("\nKey Insights:")
print(f"High-risk youth correctly identified: {cm_normalized[2,2]:.1%} (n={cm[2,2]}/{cm[2].sum()})")
print(f"Moderate-risk youth correctly identified: {cm_normalized[1,1]:.1%} (n={cm[1,1]}/{cm[1].sum()})")
print(f"High-risk misclassified as low-risk: {cm_normalized[2,0]:.1%} (n={cm[2,0]})")

model_metadata = {
    'hyperparameters':    lgbm_optimized.get_params(),
    'class_weights':      class_weights.tolist(),
    'optimal_thresholds': best_thresholds,
    'best_iteration':     lgbm_optimized.best_iteration_,
    'balanced_accuracy':  best_metrics['bal_acc'],
    'per_class_recall':   best_metrics['recall'].tolist(),
    'training_samples':   len(y_train),
    'test_samples':       len(y_test),
}
print("\nMODEL METADATA (for reproducibility)")
for key, value in model_metadata.items():
    print(f"{key}: {value}")

## 7. Project 4: Hypothesis-Driven Comparative Models

Separate models for LGBT vs heterosexual youth using a hypothesis-driven feature set. Feature names corrected to use the full renamed variable names from `df_lgbm`.

In [ ]:
# Hypothesis-driven feature set — names corrected to match df_lgbm columns
top_features = [
    # Demographics & stratification variable
    'raceeth', 'sex_iden_gender_minority', 'age_bin',
    'grade', 'Sex',

    # Violence exposure (critical for hypothesis)
    'q18_seen_phy_viol_bin',             # Witnessed violence
    'q19_forced_sex_bin',                # Forced sex
    'q20_times_forced_sex_12m_bin',      # Times forced to do sexual things
    'q88_adult_forced_sex_bin',          # Sexual abuse by adult

    # Bullying & discrimination (key LGBT stressors)
    'q24_bullied_atschool_12m_bin',      # School bullying
    'q25_cyber_bullied_12m_bin',         # Cyberbullying
    'q105_unfairly_disciplined_12m_bin', # Unfair discipline

    # Substance use
    'q31_smoked_cig_bin',                # Ever smoked
    'q35_ever_vaped_bin',                # Vapor product use
    'q40_tried_to_quit_12m_bin',         # Tried to quit tobacco
    'q41_age_1st_drink_bin',             # Age first drank
    'q44_max_drinks_30_bin',             # Max drinks in row
    'q46_times_used_marijuana_l_bin',    # Marijuana use (lifetime)
    'q47_age_1st_mari_bin',              # Age first marijuana
    'q48_times_used_marijuana_30_bin',   # Marijuana (30 days)
    'q49_times_pain_med_abuse_l_bin',    # Pain med misuse

    # ACEs (adverse childhood experiences)
    'q89_times_insulted_byparent_l_bin', # Parent insults
    'q100_lived_w_guardian_w_drugalc_issues_l_bin',      # Lived with addict
    'q101_lived_w_severely_mentally_ill_guardian_l_bin', # Lived with mentally ill parent
    'q102_separated_from_guardian_l_bin',# Parent incarceration

    # Protective factors
    'q103_feel_close_to_ppl_atschool_bin', # School connectedness
    'q76_days_phys_active_7_bin',          # Physical activity
    'q77_days_pe_class_7_bin',             # PE classes
    'q78_no_sports_team_12m_bin',          # Sports teams
    'q75_breakfast_days_bin',              # Breakfast eating
    'q85_hours_slept_bin',                 # Sleep hours

    # Health behaviors
    'q68_juice_freq_bin',    # Fruit juice
    'q69_fruit_freq_bin',    # Fruit
    'q70_salad_freq_bin',    # Green salad
    'q72_carrot_freq_bin',   # Carrots
    'q73_other_veg_freq_bin',# Vegetables
    'q74_soda_freq_bin',     # Soda
    'q80_freq_soc_media_bin',# Social media use

    # Additional substance variables
    'q32_age_1st_cig_bin',   # Age first smoked
    'q36_days_vaped_30_bin', # Vapor days
]

print("Hypothesis Testing: LGBT vs. heterosexual Mental Health Risk Model")
print(f"\nUsing {len(top_features)} hypothesis-driven features")
print(f"Total samples: {len(df_lgbm)}")

available_features = [f for f in top_features if f in df_lgbm.columns]
print(f"Features available in dataset: {len(available_features)}/{len(top_features)}")

X_hypothesis = df_lgbm[available_features].copy()
y_hypothesis = df_lgbm['mental_health_risk_category'].copy()
weights_hypothesis = df_lgbm['weight'].values

lgbt_status = df_lgbm['sex_iden_gender_minority'].copy()

lgbt_mask  = (lgbt_status == 1)
hetero_mask = ~lgbt_mask

print("Sample sizes by sexual orientation")
print(f"LGBT youth:          {lgbt_mask.sum():5d} ({lgbt_mask.mean()*100:.1f}%)")
print(f"Heterosexual youth:  {hetero_mask.sum():5d} ({hetero_mask.mean()*100:.1f}%)")

print("\nRisk category distribution within LGBT youth:")
print(y_hypothesis[lgbt_mask].value_counts().sort_index())
print(f"Proportions: {y_hypothesis[lgbt_mask].value_counts(normalize=True).sort_index().values}")

print("\nRisk category distribution within heterosexual youth:")
print(y_hypothesis[hetero_mask].value_counts().sort_index())
print(f"Proportions: {y_hypothesis[hetero_mask].value_counts(normalize=True).sort_index().values}")

# Train/test splits by group
X_lgbt   = X_hypothesis[lgbt_mask].copy()
y_lgbt   = y_hypothesis[lgbt_mask].copy()
w_lgbt   = weights_hypothesis[lgbt_mask]

X_lgbt_train, X_lgbt_test, y_lgbt_train, y_lgbt_test, w_lgbt_train, w_lgbt_test = train_test_split(
    X_lgbt, y_lgbt, w_lgbt, test_size=0.2, stratify=y_lgbt, random_state=42
)

X_hetero   = X_hypothesis[hetero_mask].copy()
y_hetero   = y_hypothesis[hetero_mask].copy()
w_hetero   = weights_hypothesis[hetero_mask]

X_hetero_train, X_hetero_test, y_hetero_train, y_hetero_test, w_hetero_train, w_hetero_test = train_test_split(
    X_hetero, y_hetero, w_hetero, test_size=0.2, stratify=y_hetero, random_state=42
)

print("\nTRAIN-TEST SPLIT")
print(f"LGBT        - Train: {len(y_lgbt_train)}, Test: {len(y_lgbt_test)}")
print(f"Heterosexual- Train: {len(y_hetero_train)}, Test: {len(y_hetero_test)}")

def compute_combined_weights(y_train, w_train, boost_factor_1=1.5, boost_factor_2=2.0):
    """Compute survey weights x class weights with boosts for minority classes."""
    classes = np.unique(y_train)
    class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights[1] *= boost_factor_1
    class_weights[2] *= boost_factor_2
    sample_class_weights = np.array([class_weights[int(y)] for y in y_train])
    w_combined = w_train * sample_class_weights
    w_combined = pd.Series(w_combined, dtype=np.float32).reset_index(drop=True)
    return w_combined, class_weights

w_lgbt_combined,   lgbt_class_weights   = compute_combined_weights(y_lgbt_train,   w_lgbt_train)
w_hetero_combined, hetero_class_weights = compute_combined_weights(y_hetero_train, w_hetero_train)
print(f"\nLGBT class weights: {lgbt_class_weights}")
print(f"Heterosexual class weights: {hetero_class_weights}")

lgbm_params = {
    'objective': 'multiclass', 'num_class': 3, 'boosting_type': 'gbdt',
    'n_estimators': 1500, 'max_depth': 10, 'learning_rate': 0.01,
    'num_leaves': 31, 'min_child_samples': 20, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'reg_alpha': 0.05, 'reg_lambda': 0.1,
    'min_split_gain': 0.02, 'min_data_per_group': 30, 'subsample_freq': 1,
    'random_state': 42, 'verbose': -1, 'n_jobs': -1
}

categorical_features = [col for col in X_lgbt_train.columns if X_lgbt_train[col].dtype.name == 'category']

print("\nTraining LGBT youth model")
lgbm_lgbt = LGBMClassifier(**lgbm_params)
lgbm_lgbt.fit(
    X_lgbt_train, y_lgbt_train,
    sample_weight=w_lgbt_combined,
    categorical_feature=categorical_features
)
y_lgbt_pred  = lgbm_lgbt.predict(X_lgbt_test)
lgbt_bal_acc = balanced_accuracy_score(y_lgbt_test, y_lgbt_pred)
print(f"LGBT model balanced accuracy: {lgbt_bal_acc:.4f}")

print("\nTraining heterosexual youth model")
lgbm_hetero = LGBMClassifier(**lgbm_params)
lgbm_hetero.fit(
    X_hetero_train, y_hetero_train,
    sample_weight=w_hetero_combined,
    categorical_feature=categorical_features
)
y_hetero_pred  = lgbm_hetero.predict(X_hetero_test)
hetero_bal_acc = balanced_accuracy_score(y_hetero_test, y_hetero_pred)
print(f"Heterosexual model balanced accuracy: {hetero_bal_acc:.4f}")

In [ ]:
# Model performance comparison
print('MODEL PERFORMANCE COMPARISON')

print('\nLGBT youth model')
print(f'balanced accuracy: {lgbt_bal_acc:.4f}')
print('\nclassification report:')
print(classification_report(y_lgbt_test, y_lgbt_pred, target_names=['low risk', 'moderate risk', 'high risk']))

print('\nconfusion matrix:')
cm_lgbt      = confusion_matrix(y_lgbt_test, y_lgbt_pred)
cm_lgbt_norm = cm_lgbt.astype('float') / cm_lgbt.sum(axis=1)[:, np.newaxis]
print('          predicted')
print('          low   mod   high')
for i, label in enumerate(['low  ', 'mod  ', 'high ']):
    print(f'true {label}  {cm_lgbt_norm[i,0]:.2f}  {cm_lgbt_norm[i,1]:.2f}  {cm_lgbt_norm[i,2]:.2f}  (n={cm_lgbt[i].sum()})')

print('\nheterosexual youth model')
print(f'balanced accuracy: {hetero_bal_acc:.4f}')
print('\nclassification report:')
print(classification_report(y_hetero_test, y_hetero_pred, target_names=['low risk', 'moderate risk', 'high risk']))

print('\nconfusion matrix:')
cm_hetero      = confusion_matrix(y_hetero_test, y_hetero_pred)
cm_hetero_norm = cm_hetero.astype('float') / cm_hetero.sum(axis=1)[:, np.newaxis]
print('          predicted')
print('          low   mod   high')
for i, label in enumerate(['low  ', 'mod  ', 'high ']):
    print(f'true {label}  {cm_hetero_norm[i,0]:.2f}  {cm_hetero_norm[i,1]:.2f}  {cm_hetero_norm[i,2]:.2f}  (n={cm_hetero[i].sum()})')

print('\nFEATURE IMPORTANCE COMPARISON (hypothesis test)')

lgbt_importance_df = pd.DataFrame({
    'feature':        X_lgbt_train.columns,
    'importance_lgbt': lgbm_lgbt.feature_importances_
})
hetero_importance_df = pd.DataFrame({
    'feature':          X_hetero_train.columns,
    'importance_hetero': lgbm_hetero.feature_importances_
})

importance_comparison = lgbt_importance_df.merge(hetero_importance_df, on='feature')
importance_comparison['absolute_diff'] = (
    importance_comparison['importance_lgbt'] - importance_comparison['importance_hetero']
)
importance_comparison['relative_diff'] = (
    (importance_comparison['importance_lgbt'] - importance_comparison['importance_hetero']) /
    (importance_comparison['importance_hetero'] + 1e-10) * 100
)
importance_comparison = importance_comparison.sort_values('absolute_diff', ascending=False)

print('\ntop 15 features more important for lgbt youth')
print(importance_comparison.head(15)[['feature', 'importance_lgbt', 'importance_hetero', 'absolute_diff']].to_string(index=False))

print('\ntop 15 features more important for heterosexual youth')
print(importance_comparison.tail(15)[['feature', 'importance_lgbt', 'importance_hetero', 'absolute_diff']].to_string(index=False))

## 8. Project 5: MICE Imputation and Advanced Modeling

Multiple Imputation by Chained Equations (MICE) to address informative missingness, followed by domain-pruned feature selection and group-stratified model fitting.

### 8.1 MICE Helper Functions

In [ ]:
label_cols = ["grade_label", "sex_label", "raceeth_label", "sex_iden_gender_minority_label"]
outcome_col = "mental_health_risk_category"
weight_col  = "weight"
group_col   = "sex_iden_gender_minority"

tag_lookup = {v["Variable Name"]: v["Tag"] for v in variable_data_renamed}

def add_missingness_blocks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add block-level missingness indicators for key domains.
    Used as MNAR probes and as predictors in imputation.
    """
    df = df.copy()

    sexual_violence = [
        "q19_forced_sex_bin", "q20_times_forced_sex_12m_bin",
        "q21_times_part_forced_sex_12m_bin", "q22_times_part_hurt_12m_bin",
        "q88_adult_forced_sex_bin",
    ]
    opioid_block = [
        "q49_times_pain_med_abuse_l_bin", "q92_times_pain_med_abuse_30_bin"
    ]
    school_connectedness = [
        "q103_feel_close_to_ppl_atschool_bin", "q103_connected",
        "q99_basic_needs_met_l_bin", "q99_high_support",
        "q104_adult_knew_whereabout_bin", "q104_high_supervision",
    ]

    blocks = {
        "miss_block_sexual_violence": sexual_violence,
        "miss_block_opioids":         opioid_block,
        "miss_block_school_support":  school_connectedness,
    }

    for new_col, cols in blocks.items():
        existing = [c for c in cols if c in df.columns]
        if not existing:
            continue
        mask_all_missing = df[existing].isna().all(axis=1)
        df[new_col] = mask_all_missing.astype("int8")

    return df

def get_predictor_columns(df: pd.DataFrame) -> List[str]:
    """
    Analysis predictors: all numeric, excluding outcome, weight, label columns.
    """
    base_exclude = set(label_cols + [outcome_col, weight_col])
    return [
        c for c in df.columns
        if c not in base_exclude and np.issubdtype(df[c].dtype, np.number)
    ]

def get_imputation_vars(df: pd.DataFrame) -> List[str]:
    """
    Variables for imputation models: predictors + outcome + weight + key design vars.
    """
    predictors = get_predictor_columns(df)
    design_vars = ["age_bin", "grade", "Sex", "raceeth", group_col]
    extra = [outcome_col, weight_col] + design_vars
    return list(dict.fromkeys(predictors + [c for c in extra if c in df.columns]))

def enforce_substantive_constraints(df: pd.DataFrame) -> pd.DataFrame:
    """
    Enforce logical constraints between related items so imputations remain plausible.
    Examples:
        - Lifetime marijuana == 0  ->  30-day marijuana must be 0
        - Lifetime pain meds == 0  ->  30-day pain meds must be 0
        - Any drunk driving == 1   ->  freq >= 1
        - High-support/grades/dental consistency
    """
    df = df.copy()

    mj_life, mj_30 = "q46_times_used_marijuana_l_bin", "q48_times_used_marijuana_30_bin"
    if {mj_life, mj_30}.issubset(df.columns):
        df.loc[df[mj_life] == 0, mj_30] = 0
        mask_30_pos = df[mj_30] > 0
        df.loc[mask_30_pos, mj_life] = np.maximum(df.loc[mask_30_pos, mj_life].fillna(0), 1)

    pain_life, pain_30 = "q49_times_pain_med_abuse_l_bin", "q92_times_pain_med_abuse_30_bin"
    if {pain_life, pain_30}.issubset(df.columns):
        df.loc[df[pain_life] == 0, pain_30] = 0
        mask_30_pain_pos = df[pain_30] > 0
        df.loc[mask_30_pain_pos, pain_life] = np.maximum(df.loc[mask_30_pain_pos, pain_life].fillna(0), 1)

    if {"q9_times_rode_w_drunkdri_30_bin", "q9_rode_wdrunkdriver_any"}.issubset(df.columns):
        df.loc[df["q9_times_rode_w_drunkdri_30_bin"] == 0, "q9_rode_wdrunkdriver_any"] = 0
        mask_any = df["q9_rode_wdrunkdriver_any"] == 1
        df.loc[mask_any, "q9_times_rode_w_drunkdri_30_bin"] = np.maximum(
            df.loc[mask_any, "q9_times_rode_w_drunkdri_30_bin"].fillna(0), 1
        )

    if {"q99_basic_needs_met_l_bin", "q99_high_support"}.issubset(df.columns):
        mask_high = df["q99_high_support"] == 1
        df.loc[mask_high, "q99_basic_needs_met_l_bin"] = np.maximum(
            df.loc[mask_high, "q99_basic_needs_met_l_bin"].fillna(0), 2
        )

    if {"q87_grades_12m_bin", "q87_high_grades"}.issubset(df.columns):
        df.loc[df["q87_high_grades"] == 1, "q87_grades_12m_bin"] = 0

    if {"q83_last_dental_visit_bin", "q83_recent_dental"}.issubset(df.columns):
        df.loc[df["q83_recent_dental"] == 1, "q83_last_dental_visit_bin"] = 0

    return df

def weighted_chi2(df: pd.DataFrame, x: str, y: str, w: str = weight_col) -> float:
    """
    Pseudo survey-weighted chi-square statistic for association
    between categorical predictor x and categorical outcome y.
    """
    sub = df[[x, y, w]].dropna()
    if sub.empty:
        return np.nan

    ct = pd.pivot_table(sub, index=x, columns=y, values=w, aggfunc="sum", fill_value=0.0)
    observed = ct.values.astype(float)
    row_sums  = observed.sum(axis=1, keepdims=True)
    col_sums  = observed.sum(axis=0, keepdims=True)
    total     = observed.sum()
    if total == 0:
        return np.nan

    expected = row_sums * col_sums / total
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = ((observed - expected) ** 2 / np.where(expected == 0, 1, expected)).sum()
    return float(chi2)

def rank_predictors_by_association(
    df: pd.DataFrame,
    features: List[str],
    outcome: str = outcome_col,
    weight: str = weight_col,
) -> pd.DataFrame:
    """
    Computes survey-weighted chi-square score for each feature vs outcome.
    Returns a ranked DataFrame.
    """
    rows = []
    for col in features:
        if col in (outcome, weight): continue
        try:
            chi2 = weighted_chi2(df, col, outcome, weight)
        except Exception:
            chi2 = np.nan
        rows.append({"variable": col, "chi2": chi2})

    res = pd.DataFrame(rows).dropna().sort_values("chi2", ascending=False)
    res["rank"] = np.arange(1, len(res) + 1)
    return res

def run_mice(df: pd.DataFrame, n_imputations: int = 8, random_state: int = 42) -> List[pd.DataFrame]:
    """
    Multiple imputation via IterativeImputer:
      - Uses all numeric predictors + outcome + weight + design vars
      - Includes SGM indicator
      - Uses BayesianRidge (supports return_std) for sample_posterior draws
      - Enforces substantive constraints after imputation
    """
    df = df.copy()
    impute_cols = get_imputation_vars(df)
    X = df[impute_cols]

    base_estimator = BayesianRidge()
    imputer = IterativeImputer(
        estimator=base_estimator, max_iter=8, sample_posterior=True, random_state=random_state
    )

    imputations = []
    start_time = time.time()
    for k in tqdm(range(n_imputations), desc="Imputations", unit="imp"):
        imputer.random_state = random_state + k
        imputed_array = imputer.fit_transform(X)
        imputed_df = df.copy()
        imputed_df[impute_cols] = imputed_array

        for col in impute_cols:
            if pd.api.types.is_integer_dtype(df[col].dropna()):
                imputed_df[col] = np.round(imputed_df[col]).astype(int)

        imputed_df = enforce_substantive_constraints(imputed_df)
        imputations.append(imputed_df)

    elapsed = time.time() - start_time
    print(f"Imputation completed in {elapsed:.1f} seconds for {n_imputations} datasets.")
    return imputations

def impute_and_rank(df: pd.DataFrame, n_imputations: int = 8) -> Dict[str, pd.DataFrame]:
    """
    Full pipeline:
      1. Adds missingness-block indicators.
      2. Runs multiple imputations.
      3. In each imputed dataset, computes survey-weighted chi2 for each predictor.
      4. Averages ranks and chi2 across imputations; attach domain tags.

    Returns:
        {
            "per_imputation": long-format rank table,
            "summary": averaged ranks with tags,
            "imputed_datasets": list of imputed DataFrames
        }
    """
    df_aug = add_missingness_blocks(df)
    imputed_dfs = run_mice(df_aug, n_imputations=n_imputations)

    all_scores = []
    start_time = time.time()
    for m, imp_df in enumerate(tqdm(imputed_dfs, desc="Ranking features", unit="imp")):
        features = get_predictor_columns(imp_df)
        ranks = rank_predictors_by_association(imp_df, features)
        ranks["m"] = m
        all_scores.append(ranks)

    elapsed = time.time() - start_time
    print(f"Feature ranking completed in {elapsed:.1f} seconds across {n_imputations} imputations.")

    all_scores_df = pd.concat(all_scores, ignore_index=True)

    summary = (
        all_scores_df
        .groupby("variable", as_index=False)
        .agg(mean_rank=("rank", "mean"), mean_chi2=("chi2", "mean"), n_imputations=("m", "nunique"))
        .sort_values("mean_rank")
    )
    summary["Tag"] = summary["variable"].map(tag_lookup).fillna("Unknown")

    return {
        "per_imputation":  all_scores_df,
        "summary":         summary,
        "imputed_datasets": imputed_dfs,
    }

results          = impute_and_rank(df_encoded_renamed, n_imputations=8)
summary          = results["summary"]
imputed_datasets = results["imputed_datasets"]

### 8.2 Domain Pruning

In [ ]:
def prune_by_domain(
    summary: pd.DataFrame,
    min_keep_per_domain: int = 2,
    max_drop_frac_per_domain: float = 0.5,
) -> Dict[str, List[str]]:
    """
    Domain-wise pruning based on the summary table.

    Args:
        summary: DataFrame from impute_and_rank()["summary"]
        min_keep_per_domain: minimum variables to keep per Tag/domain
        max_drop_frac_per_domain: max fraction dropped per Tag

    Returns:
        dict with "keep", "drop", "by_domain" keys
    """
    df = summary.copy()
    if "Tag" not in df.columns:
        df["Tag"] = "Unknown"

    tags = sorted(df["Tag"].unique())
    n_domains = len(tags)

    keep_vars, drop_vars, records = [], [], []

    start_time = time.time()
    print(f"Pruning domains: {n_domains} domains to process")

    for tag, sub in tqdm(df.groupby("Tag"), total=n_domains, desc="Domain pruning"):
        sub_sorted = sub.sort_values("mean_rank")
        n_total  = len(sub_sorted)
        max_drop = int(np.floor(max_drop_frac_per_domain * n_total))
        n_keep   = max(min_keep_per_domain, n_total - max_drop)

        keep = sub_sorted["variable"].iloc[:n_keep].tolist()
        drop = sub_sorted["variable"].iloc[n_keep:].tolist()
        keep_vars.extend(keep)
        drop_vars.extend(drop)
        records.append({"Tag": tag, "n_total": n_total, "n_keep": len(keep), "n_drop": len(drop)})

    by_domain = pd.DataFrame(records)
    elapsed   = time.time() - start_time
    print(f"Domain pruning completed in {elapsed:.3f} seconds")

    return {"keep": keep_vars, "drop": drop_vars, "by_domain": by_domain}

pruning   = prune_by_domain(summary, min_keep_per_domain=2, max_drop_frac_per_domain=0.5)
vars_keep = pruning["keep"]
vars_drop = pruning["drop"]
pruning["by_domain"]

### 8.3 Observed vs Imputed Diagnostics

In [ ]:
def compare_obs_imputed_by_group(
    original_df: pd.DataFrame,
    imputed_dfs: List[pd.DataFrame],
    vars_to_check: List[str],
    race_col: str = "raceeth",
    sgm_col:  str = "sex_iden_gender_minority",
    weight_col: str = "weight",
) -> pd.DataFrame:
    """
    Compare observed vs imputed distributions by race and SGM subgroup.
    Vectorized approach for speed across many variables.
    """
    orig  = original_df.copy()
    rows  = []
    races = sorted(orig[race_col].dropna().unique())
    sgms  = sorted(orig[sgm_col].dropna().unique())

    start_time = time.time()
    valid_vars  = [v for v in vars_to_check if v in orig.columns]
    pbar        = tqdm(valid_vars, desc="Comparing obs vs imputed", unit="var")

    for idx, var in enumerate(pbar, start=1):
        cats = set(orig[var].dropna().unique())
        for imp_df in imputed_dfs:
            if var in imp_df.columns:
                cats |= set(imp_df[var].dropna().unique())
        cats = sorted([c for c in cats if pd.notna(c)])

        obs             = orig[[var, race_col, sgm_col, weight_col]].dropna(subset=[race_col, sgm_col, weight_col])
        obs_nonmissing  = obs.dropna(subset=[var])
        if obs_nonmissing.empty:
            continue

        obs_group_totals = obs_nonmissing.groupby([race_col, sgm_col])[weight_col].sum().rename("w_tot_obs").reset_index()
        obs_ct           = obs_nonmissing.groupby([race_col, sgm_col, var])[weight_col].sum().rename("w_cat_obs").reset_index()
        obs_merged       = obs_ct.merge(obs_group_totals, on=[race_col, sgm_col], how="left")
        obs_merged["prop_obs"] = obs_merged["w_cat_obs"] / obs_merged["w_tot_obs"]

        obs_grid = (
            pd.MultiIndex.from_product([races, sgms, cats], names=[race_col, sgm_col, "category"])
            .to_frame(index=False)
        )
        obs_grid = obs_grid.merge(
            obs_merged.rename(columns={var: "category"})[[race_col, sgm_col, "category", "prop_obs"]],
            on=[race_col, sgm_col, "category"], how="left",
        )

        imp_cat_counts, imp_totals = {}, {}
        for imp_df in imputed_dfs:
            if var not in imp_df.columns: continue
            sub           = imp_df[[var, race_col, sgm_col, weight_col]].dropna(subset=[race_col, sgm_col, weight_col])
            sub_nonmissing= sub.dropna(subset=[var])
            if sub_nonmissing.empty: continue
            grp_tot = sub_nonmissing.groupby([race_col, sgm_col])[weight_col].sum()
            grp_ct  = sub_nonmissing.groupby([race_col, sgm_col, var])[weight_col].sum()
            for (r, g), w_tot in grp_tot.items():
                imp_totals[(r, g)] = imp_totals.get((r, g), 0.0) + w_tot
            for (r, g, cat), w_cat in grp_ct.items():
                imp_cat_counts[(r, g, cat)] = imp_cat_counts.get((r, g, cat), 0.0) + w_cat

        n_imp = sum(1 for imp_df in imputed_dfs if var in imp_df.columns)
        imp_grid = obs_grid.copy()
        imp_grid["w_tot_imp"] = imp_grid.apply(lambda row: imp_totals.get((row[race_col], row[sgm_col]), np.nan), axis=1)
        imp_grid["w_cat_imp"] = imp_grid.apply(lambda row: imp_cat_counts.get((row[race_col], row[sgm_col], row["category"]), np.nan), axis=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            imp_grid["prop_imp"] = np.where(
                (imp_grid["w_tot_imp"] > 0) & np.isfinite(imp_grid["w_cat_imp"]),
                imp_grid["w_cat_imp"] / imp_grid["w_tot_imp"], np.nan,
            )

        merged = imp_grid.merge(obs_grid[[race_col, sgm_col, "category", "prop_obs"]], on=[race_col, sgm_col, "category"], how="left", suffixes=("", "_obs"))

        for _, row in merged.iterrows():
            rows.append({
                "variable": var, "category": row["category"],
                "race": row[race_col], "sgm": row[sgm_col],
                "prop_obs": row["prop_obs"], "prop_imp": row["prop_imp"], "n_imp": n_imp,
            })

        elapsed     = time.time() - start_time
        avg_per_var = elapsed / idx
        remaining   = avg_per_var * (len(valid_vars) - idx)
        pbar.set_postfix({"elapsed_s": f"{elapsed:6.1f}", "eta_s": f"{remaining:6.1f}"})

    pbar.close()
    print(f"Completed comparison for {len(valid_vars)} variables in {time.time()-start_time:.1f} seconds.")
    return pd.DataFrame(rows)

vars_to_check = [
    "q19_forced_sex_bin",
    "q49_times_pain_med_abuse_l_bin",
    "q103_feel_close_to_ppl_atschool_bin",
]
diag = compare_obs_imputed_by_group(
    original_df=df_encoded_renamed,
    imputed_dfs=imputed_datasets,
    vars_to_check=vars_to_check,
)

### 8.4 Restrict Predictors for Modeling

In [ ]:
def restrict_predictors_for_modeling(
    imputed_dfs: List[pd.DataFrame],
    vars_keep: List[str],
) -> List[pd.DataFrame]:
    """
    For each imputed dataset, keep only vars_keep plus core variables
    (outcome, weight, group, design vars).
    """
    start_time  = time.time()
    model_dfs   = []
    core_keep   = {outcome_col, weight_col, group_col, "age_bin", "grade", "Sex", "raceeth"}
    vars_keep_set = set(vars_keep) | core_keep

    for df_imp in tqdm(imputed_dfs, desc="Restricting predictors for modeling"):
        cols_to_keep = [c for c in df_imp.columns if c in vars_keep_set]
        model_dfs.append(df_imp[cols_to_keep].copy())

    print(f"Finished restricting predictors in {time.time()-start_time:.2f} seconds.")
    return model_dfs

model_datasets = restrict_predictors_for_modeling(imputed_datasets, vars_keep)

### 8.5 Model Fitting Utilities

In [ ]:
def get_model_predictors(df: pd.DataFrame) -> List[str]:
    """Return predictor columns for modeling (after pruning): numeric, excluding outcome/weight/group/labels."""
    base_exclude = set(label_cols + [outcome_col, weight_col, group_col])
    return [
        c for c in df.columns
        if c not in base_exclude and np.issubdtype(df[c].dtype, np.number)
    ]

def make_class_weights(y: np.ndarray) -> Dict[int, float]:
    """Simple inverse-frequency class weights for 3-class risk outcome."""
    classes, counts = np.unique(y, return_counts=True)
    total   = counts.sum()
    weights = {int(c): float(total / (len(classes) * cnt)) for c, cnt in zip(classes, counts)}
    return weights

### 8.6 Multinomial Elastic-Net

In [ ]:
def fit_multinomial_elastic_net(
    df: pd.DataFrame,
    group_value: int,
    n_splits: int = 3,
    random_state: int = 42,
):
    """
    Fit survey-weighted, class-weighted multinomial logistic regression
    with elastic-net penalty for a given group (LGBT vs majority).

    Returns:
        best_estimator, cv_results (macro-F1, high-risk recall)
    """
    n_splits = min(n_splits, 3)
    n_param_combinations = 2 * 2
    total_fits = n_param_combinations * n_splits

    progress   = tqdm(total=total_fits, desc=f"Group {group_value} CV", unit="fit")
    start_time = time.time()

    def _update_progress():
        progress.n = total_fits
        progress.refresh()

    sub    = df[df[group_col] == group_value].copy()
    X_cols = get_model_predictors(sub)
    X      = sub[X_cols].values
    y      = sub[outcome_col].values
    w      = sub[weight_col].values

    class_w            = make_class_weights(y)
    class_weight_array = np.array([class_w[int(c)] for c in y])
    sample_weight      = w * class_weight_array

    pipe = Pipeline(steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LogisticRegression(
            penalty="elasticnet", solver="saga", max_iter=2000,
            multi_class="multinomial", n_jobs=-1,
        )),
    ])

    param_grid = {"clf__C": [0.5, 1.0], "clf__l1_ratio": [0.5]}

    def macro_f1_highrisk_recall(est, X_val, y_val):
        y_pred    = est.predict(X_val)
        macro_f1  = f1_score(y_val, y_pred, average="macro")
        high_mask = (y_val == 2)
        rec_high  = recall_score(y_val, y_pred, labels=[2], average="macro") if high_mask.sum() > 0 else 0.0
        return 0.5 * macro_f1 + 0.5 * rec_high

    scorer = make_scorer(macro_f1_highrisk_recall, greater_is_better=True)
    cv     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    grid = GridSearchCV(estimator=pipe, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1, refit=True, verbose=2)
    grid.fit(X, y, clf__sample_weight=sample_weight)

    _update_progress()
    progress.close()
    print(f"Group {group_value} grid search completed in {(time.time()-start_time)/60:.2f} minutes.")

    return grid.best_estimator_, grid

### 8.7 LightGBM with SHAP

In [ ]:
def fit_lgbm_with_shap(
    df: pd.DataFrame,
    group_value: int,
    n_splits: int = 3,
    random_state: int = 42,
):
    """
    Fit LightGBM classifier with survey and class weights for a given group.
    Compute global SHAP values for feature importance.

    Returns:
        best_estimator, shap_values, feature_names
    """
    sub    = df[df[group_col] == group_value].copy()
    X_cols = get_model_predictors(sub)
    X      = sub[X_cols].values
    y      = sub[outcome_col].values
    w      = sub[weight_col].values

    class_w            = make_class_weights(y)
    class_weight_array = np.array([class_w[int(c)] for c in y])
    sample_weight      = w * class_weight_array

    base_model = LGBMClassifier(
        objective="multiclass", num_class=3, n_estimators=50, learning_rate=0.08,
        num_leaves=31, max_depth=-1, subsample=0.8, colsample_bytree=0.8,
        random_state=random_state, n_jobs=-1,
    )

    param_grid = {"min_child_samples": [20, 50], "reg_lambda": [0.0]}
    total_param_combos = len(param_grid["min_child_samples"]) * len(param_grid["reg_lambda"])
    total_cv_fits      = total_param_combos * n_splits

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    def macro_f1_highrisk_recall(est, X_val, y_val):
        y_pred    = est.predict(X_val)
        macro_f1  = f1_score(y_val, y_pred, average="macro")
        high_mask = (y_val == 2)
        rec_high  = recall_score(y_val, y_pred, labels=[2], average="macro") if high_mask.sum() > 0 else 0.0
        return 0.5 * macro_f1 + 0.5 * rec_high

    scorer = make_scorer(macro_f1_highrisk_recall, greater_is_better=True)

    print(f"[LightGBM] Group={group_value} | Hyperparam combos: {total_param_combos}, "
          f"CV folds: {n_splits}, total fits: {total_cv_fits}")

    t0 = time.time()
    grid = GridSearchCV(estimator=base_model, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1, refit=True, verbose=0)
    grid.fit(X, y, sample_weight=sample_weight)
    elapsed = time.time() - t0

    best_model = grid.best_estimator_
    explainer  = shap.TreeExplainer(best_model)

    max_shap_n = 3000
    X_shap     = X[:max_shap_n] if X.shape[0] > max_shap_n else X
    shap_values = explainer.shap_values(X_shap)

    return best_model, shap_values, X_cols

### 8.8 Evaluate Across Imputations

In [ ]:
def evaluate_models_across_imputations(
    model_datasets: List[pd.DataFrame],
    groups=(0, 1),
    random_state: int = 42,
):
    """
    For each imputed dataset and group:
      - Fit multinomial elastic-net
      - Fit LightGBM + SHAP
    Returns a nested dict of fitted models and metrics per imputation and group.
    Pooling of metrics can then be done by simple averaging and variance across imputations.
    """
    results       = {"logit": {}, "lgbm": {}}
    n_imputations = len(model_datasets)
    n_groups      = len(groups)
    total_iters   = n_imputations * n_groups
    start_time    = time.time()

    with tqdm(total=total_iters, desc="Model fits (all imputations x groups)") as pbar:
        iter_idx = 0
        for m, df_imp in enumerate(model_datasets):
            results["logit"][m] = {}
            results["lgbm"][m]  = {}
            for g in groups:
                iter_idx += 1
                pbar.set_description(f"Imputation {m+1}/{n_imputations} | Group {g} | Iter {iter_idx}/{total_iters}")

                logit_model, logit_grid = fit_multinomial_elastic_net(df_imp, group_value=g, random_state=random_state+m)
                results["logit"][m][g] = {
                    "model":       logit_model,
                    "cv_results":  logit_grid.cv_results_,
                    "best_params": logit_grid.best_params_,
                    "best_score":  logit_grid.best_score_,
                }

                lgbm_model, shap_vals, feat_names = fit_lgbm_with_shap(df_imp, group_value=g, random_state=random_state+m)
                results["lgbm"][m][g] = {
                    "model":         lgbm_model,
                    "shap_values":   shap_vals,
                    "feature_names": feat_names,
                    "best_params":   lgbm_model.get_params(),
                }

                pbar.update(1)
                pbar.set_postfix({"elapsed_min": f"{(time.time()-start_time)/60:.1f}"})

    return results

model_results = evaluate_models_across_imputations(model_datasets)

### 8.9 Stratified Split

In [ ]:
def make_stratified_split_indices(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 123):
    """Make a single stratified train/test split index on outcome."""
    y = df[outcome_col].values
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(df, y))
    return train_idx, test_idx

train_idx, test_idx = make_stratified_split_indices(model_datasets[0])

### 8.10 Holdout Evaluation

In [ ]:
def evaluate_on_holdout(
    df: pd.DataFrame,
    model_logit,
    model_lgbm,
    train_idx,
    test_idx,
    group_value: int,
):
    """
    Evaluate logit and LightGBM models on a holdout split within one imputed dataset
    and one group.
    """
    sub    = df[df[group_col] == group_value].copy()
    X_cols = get_model_predictors(sub)
    sub    = sub.reset_index(drop=True)

    X = sub[X_cols].values
    y = sub[outcome_col].values

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123)
    train_g, test_g = next(splitter.split(X, y))
    X_test  = X[test_g]
    y_test  = y[test_g]

    y_pred_logit = model_logit.predict(X_test)
    cm_logit     = confusion_matrix(y_test, y_pred_logit, labels=[0, 1, 2])
    macro_f1_logit  = f1_score(y_test, y_pred_logit, average="macro")
    rec_high_logit  = recall_score(y_test, y_pred_logit, labels=[2], average="macro")

    y_pred_lgbm  = model_lgbm.predict(X_test)
    cm_lgbm      = confusion_matrix(y_test, y_pred_lgbm, labels=[0, 1, 2])
    macro_f1_lgbm   = f1_score(y_test, y_pred_lgbm, average="macro")
    rec_high_lgbm   = recall_score(y_test, y_pred_lgbm, labels=[2], average="macro")

    metrics = {
        "logit": {"macro_f1": macro_f1_logit, "recall_high": rec_high_logit, "confusion_matrix": cm_logit},
        "lgbm":  {"macro_f1": macro_f1_lgbm,  "recall_high": rec_high_lgbm,  "confusion_matrix": cm_lgbm},
    }
    return metrics, X_test, y_test

def plot_confusion(cm, title):

    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Low", "Mod", "High"],
                yticklabels=["Low", "Mod", "High"])
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.tight_layout(); plt.show()

### 8.11 Evaluation Execution

In [ ]:
# Evaluate imputation 0 for both groups
m = 0
df_imp = model_datasets[m]

logit_majority = model_results["logit"][m][0]["model"]
lgbm_majority  = model_results["lgbm"][m][0]["model"]

metrics_maj, X_test_maj, y_test_maj = evaluate_on_holdout(
    df_imp, logit_majority, lgbm_majority, train_idx, test_idx, group_value=0
)
plot_confusion(metrics_maj["logit"]["confusion_matrix"], "Logit Majority")
plot_confusion(metrics_maj["lgbm"]["confusion_matrix"],  "LGBM Majority")

logit_lgbt = model_results["logit"][m][1]["model"]
lgbm_lgbt  = model_results["lgbm"][m][1]["model"]

metrics_lgbt, X_test_lgbt, y_test_lgbt = evaluate_on_holdout(
    df_imp, logit_lgbt, lgbm_lgbt, train_idx, test_idx, group_value=1
)
plot_confusion(metrics_lgbt["logit"]["confusion_matrix"], "Logit LGBT")
plot_confusion(metrics_lgbt["lgbm"]["confusion_matrix"],  "LGBM LGBT")

print("Majority logit:", metrics_maj["logit"]["macro_f1"], metrics_maj["logit"]["recall_high"])
print("Majority lgbm: ", metrics_maj["lgbm"]["macro_f1"],  metrics_maj["lgbm"]["recall_high"])
print("LGBT logit   :", metrics_lgbt["logit"]["macro_f1"], metrics_lgbt["logit"]["recall_high"])
print("LGBT lgbm    :", metrics_lgbt["lgbm"]["macro_f1"],  metrics_lgbt["lgbm"]["recall_high"])